<div style="font-family: 'Segoe UI', Roboto, 'Helvetica Neue', Arial, sans-serif; background: linear-gradient(120deg, #0b1f4b 0%, #173f73 50%, #1d5aa6 100%); padding: 30px 32px 26px 32px; border-radius: 14px; color: #ffffff;">
<div style="font-size: 13px; letter-spacing: 3px; text-transform: uppercase; color: #ffffff;">Google Quantum AI &middot; Verifiable Quantum Circuit Compilation</div>
<div style="font-size: 34px; font-weight: 800; line-height: 1.15; margin-top: 8px; color: #ffffff;">What we tried, and why it failed</div>
<div style="font-size: 18px; margin-top: 8px; color: #ffffff;">Sixteen measured ideas on the best public circuits, the four that survived the verifier, the method we use to hunt for gains, and an exact simulator you can reuse</div>
<div style="margin-top: 16px;">
<span style="background: #ffd166; color: #0b1f4b; border-radius: 12px; padding: 4px 12px; font-size: 13px; font-weight: 700; margin-right: 6px;">same submission as our best notebook: GM 27.307 (local, default salt)</span>
<span style="background: #ffffff; color: #0b1f4b; border-radius: 12px; padding: 4px 12px; font-size: 13px; font-weight: 700; margin-right: 6px;">invsqrt</span>
<span style="background: #ffffff; color: #0b1f4b; border-radius: 12px; padding: 4px 12px; font-size: 13px; font-weight: 700; margin-right: 6px;">DQI</span>
<span style="background: #ffffff; color: #0b1f4b; border-radius: 12px; padding: 4px 12px; font-size: 13px; font-weight: 700; margin-right: 6px;">chemistry</span>
</div>
</div>

<div style="font-family: 'Segoe UI', Roboto, 'Helvetica Neue', Arial, sans-serif; font-size: 15.5px; line-height: 1.55; color: #111111; background: #ffffff; border: 1px solid #c9d6ea; border-radius: 10px; padding: 14px 18px; margin-top: 10px;">
<div style="font-size: 19px; font-weight: 700; color: #0b1f4b; margin-bottom: 6px; background: #ffffff;">Five findings, with the numbers</div>
<ol style="margin: 0; padding-left: 22px; color: #111111; background: #ffffff;">
<li><b>One Toffoli was hiding in the best public invsqrt circuit.</b> A Toffoli whose target is only measured away can be replaced by measuring first and a free classically controlled CZ. Official verifier: 78,765.51 &rarr; <b>78,719.53</b>, proved on all 3,133,452 inputs. Our analysis library <code style="background: #eef1f6; color: #111111; padding: 1px 5px; border-radius: 4px; font-size: 92%;">qccscope</code> found it in 6 seconds (section 5).</li>
<li><b>invsqrt at 45 qubits is blocked by the adder carries.</b> Three attempts: every peak at 46 wires is a carry or AND temporary; the cheapest exact way around them costs about <b>+196 Toffolis</b>, against a break-even of about <b>38</b>.</li>
<li><b>The DQI field multipliers are already at the best known bilinear rank</b> (42 for GF(2<sup>12</sup>), 99 for GF(2<sup>24</sup>)): 81% of the decoder's Toffolis, and the whole multiplier-rewrite lever is <b>27 Toffolis (0.007%)</b>.</li>
<li><b>The DQI gates that look like no-ops are live.</b> 948 of 1,020 rare routing slots fire on about 1 in 4,096 frames, and kickmix has no measurement that yields a data-dependent classical bit, so rarity cannot lower the executed Toffolis.</li>
<li><b>Chemistry is pressed against its floors.</b> The loaders sit at the counting floor of their architecture, and one column less (C = 44 &rarr; 43) at fixed angle words cannot pass the 20 mHa gate: the error floor rises to <b>21.1 mHa</b>.</li>
</ol>
</div>

In [ ]:
WRITE_SUBMISSION = True  # True: also write /kaggle/working/submission.zip (byte for byte our solution notebook's file)

In [ ]:
# Imports, the notebook's plotting style, and the first-screen figure (data: our experiment logs, cited in section 3).
import sys, os, pathlib, zipfile, json, time, re, hashlib, shutil, math, collections, csv, subprocess, heapq
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, Circle, Rectangle
from matplotlib.ticker import FixedLocator, FormatStrFormatter, FuncFormatter
T0 = time.time()
def log(*a): print(f"[{(time.time()-T0)/60:5.1f} min]", *a, flush=True)
INK, INK2, MUTED, RULE, SURF, NAVY = "#111111", "#333333", "#6b6b6b", "#d9d9d9", "#ffffff", "#0b1f4b"
C = ["#1f6fd1", "#d9541e", "#0f9a68", "#c98a00", "#d4558e", "#007a00", "#5843c9", "#d1302f"]
TC = {"invsqrt": C[0], "dqi": C[1], "dfthcsos": C[2], "chemistry": C[2]}
TN = {"invsqrt": "invsqrt", "dqi": "DQI", "dfthcsos": "chemistry"}
plt.rcParams.update({"font.family": "DejaVu Sans", "figure.facecolor": SURF, "axes.facecolor": SURF, "savefig.facecolor": SURF,
                     "axes.edgecolor": "#888888", "axes.labelcolor": INK, "xtick.color": INK, "ytick.color": INK, "text.color": INK,
                     "axes.grid": True, "grid.color": "#ececec", "grid.linewidth": 0.6, "axes.spines.top": False, "axes.spines.right": False,
                     "font.size": 11.5, "axes.titlesize": 12.5, "axes.titleweight": "bold", "axes.titlecolor": INK, "axes.labelsize": 11.5,
                     "xtick.labelsize": 11, "ytick.labelsize": 11, "legend.fontsize": 10.5, "lines.linewidth": 2, "legend.frameon": False,
                     "axes.axisbelow": True, "figure.dpi": 110})

def headline(fig, title, sub=None):
    """Figure title that states the takeaway (left aligned), with an optional dark-grey subtitle."""
    fig.text(0.012, 0.988, title, ha="left", va="top", fontsize=15.5, fontweight="bold", color=INK)
    if sub: fig.text(0.012, 0.988 - 0.36 / fig.get_figheight(), sub, ha="left", va="top", fontsize=11, color=INK2)
    top = 1 - ((0.54 + 0.2 * (sub.count("\n") + 1)) if sub else 0.46) / fig.get_figheight()
    fig.tight_layout(rect=(0, 0, 1, top))

def pts(gm, rel_cut):
    """Score points from cutting one task's cost by the fraction rel_cut (negative = the cost grows)."""
    return gm * ((1 / (1 - rel_cut)) ** (1 / 3) - 1)

GM_NOW = 27.3067          # our solution notebook, local official verifiers, default salt
INV_C, INV_Q = 1711.294, 46  # invsqrt after our fix (official receipt: 78,719.53 = 46 x 1711.294)
DQI_T, CH_LT = 382600.38, 14738.0  # DQI mean Toffolis (official); chemistry Toffolis of the block encoding (official)
inv_t = lambda dT, dQ=0: pts(GM_NOW, 1 - ((INV_Q + dQ) * (INV_C + dT)) / (INV_Q * INV_C))
dqi_t = lambda dT: pts(GM_NOW, -dT / DQI_T)
ch_lt = lambda dT_pass: pts(GM_NOW, -2 * dT_pass / CH_LT)
# (task, idea, ceiling in points if it had worked fully, measured result in points or None if nothing was built, status)
IDEAS = [
    ("invsqrt", "dead-write fix (ours, in the submission)", inv_t(-1), pts(GM_NOW, 1 - 78719.53 / 78765.51), "worked: official verifier"),
    ("invsqrt", "splice of public circuits at clean cuts (12 Nov)", pts(25.45, 0.0020), pts(25.45, 0.0020), "worked: then the 46-qubit line arrived"),
    ("invsqrt", "45 qubits (3 attempts)", inv_t(0, -1), inv_t(196, -1), "blocked: +196 Toffolis vs ~38 break-even"),
    ("invsqrt", "drop or re-tune a rotation", inv_t(-7.0), pts(GM_NOW, 1 - 79369.55 / 78765.26), "infeasible: 32 of 33 drops; 1 costs more"),
    ("invsqrt", "pebbling: skip the recomputations", inv_t(-102), 0.0, "0 avoidable at 46 qubits"),
    ("dqi", "readout windows moved onto bb4287e7 (11 Nov)", pts(25.4, 48.28 / 394165.16), pts(25.4, 48.28 / 394165.16), "worked: official verifier"),
    ("dqi", "retain 2 more products in 4c81b726", dqi_t(-1.58), dqi_t(-1.58), "worked, too small to matter"),
    ("dqi", "cheaper field multipliers", dqi_t(-27), None, "at best known rank; 27 Toffolis left"),
    ("dqi", "keep products across the return path", dqi_t(-61380), 0.0, "0 at 889 qubits (no free wire)"),
    ("dqi", "skip the rare routing 'no-ops'", dqi_t(-17812), 0.0, "live gates; impossible in kickmix"),
    ("dqi", "one qubit less (892->891, 889->888)", pts(GM_NOW, 1 - (888 / 889) ** 2), None, "plans only; nothing exact built"),
    ("dqi", "raise the extractor cutoff to 19", dqi_t(-116), dqi_t(-116) - 0.35 * 0.041, "negative: +0.35 failures per draw"),
    ("dfthcsos", "re-plan the loaders (2 planners)", ch_lt(-375), 0.0, "0: at the floor of the block method"),
    ("dfthcsos", "drop 31 'dead' loader Toffolis", ch_lt(-31), None, "rejected by the official verifier"),
    ("dfthcsos", "one column less (C 44 -> 43), refit", pts(GM_NOW, 0.0053), None, "infeasible: error floor 21.1 mHa > 20"),
    ("dfthcsos", "one group less (R 8 -> 7), refit", pts(GM_NOW, 0.029), None, "infeasible: error 256 mHa vs 20"),
]

def first_screen_figure():
    lin = 1e-3
    tr = lambda x: np.sign(x) * np.log10(1 + abs(x) / lin)
    fig, ax = plt.subplots(figsize=(12.5, 7.8))
    n = len(IDEAS)
    for i, (task, idea, ceil, got, status) in enumerate(IDEAS):
        col = TC[task]
        ax.barh(i, tr(ceil), color=col, alpha=0.18, height=0.72, edgecolor=col, linewidth=1.0)
        if got is None:
            ax.scatter([0], [i], marker="x", s=75, color=INK, zorder=4, linewidths=2.2)
        else:
            ax.scatter([tr(got)], [i], s=95, color=col if got > 0 else (C[7] if got < 0 else INK), edgecolor="white", linewidth=1.4, zorder=4)
        ax.text(tr(2.2), i, status, va="center", fontsize=10.5, color="#0f7a4f" if status.startswith("worked") else INK,
                fontweight="bold" if status.startswith("worked") else "normal")
    ax.axvline(0, color=INK, lw=1.2)
    for yb in (4.5, 11.5): ax.axhline(yb, color="#9a9a9a", lw=0.8)
    ticks = [-1, -0.1, -0.01, 0, 0.01, 0.1, 1]
    ax.set_xticks([tr(t) for t in ticks]); ax.set_xticklabels([f"{t:+g}" if t else "0" for t in ticks])
    ax.set_xlim(tr(-1.6), tr(4e4)); ax.set_ylim(n - 0.4, -0.6)
    ax.set_yticks(range(n)); ax.set_yticklabels([idea for _, idea, *_ in IDEAS], fontsize=11)
    for lab, t in zip(ax.get_yticklabels(), IDEAS): lab.set_color(TC[t[0]]); lab.set_fontweight("bold" if t[4].startswith("worked") else "normal")
    ax.grid(axis="y", visible=False); ax.spines["left"].set_visible(False); ax.tick_params(axis="y", length=0)
    ax.set_xlabel("leaderboard points (symmetric log scale)", x=0.3)
    headline(fig, "Big ceilings, small wins: 16 ideas measured, 4 survived the verifier",
             "pale bar = what the idea was worth if it fully worked;  dot = what we measured;  x = nothing exact could be built\n"
             "blue = invsqrt, orange = DQI, green = chemistry;  each row is one of our experiments (cards in section 3)")
    plt.show()
first_screen_figure()

<div style="font-family: 'Segoe UI', Roboto, 'Helvetica Neue', Arial, sans-serif; font-size: 15.5px; line-height: 1.6; color: #111111; background: #eef4fc; border-left: 6px solid #1f6fd1; padding: 12px 18px; border-radius: 8px;">
<div style="font-size: 18px; font-weight: 700; color: #0b1f4b; margin-bottom: 4px; background: #ffffff;">Contents</div>
<a href="#ack" style="color: #0b3d91; background: #ffffff;">0. Acknowledgements and who did what</a><br>
<a href="#s1" style="color: #0b3d91; background: #ffffff;">1. The competition in five minutes</a> &mdash; tasks, score, verifiers, the one trick everyone uses<br>
<a href="#s2" style="color: #0b3d91; background: #ffffff;">2. Where the cost goes, and what one qubit or Toffoli is worth</a><br>
<a href="#s4" style="color: #0b3d91; background: #ffffff;"><b>3. What we tried and why it failed</b></a> &mdash; sixteen cards, four walls, and what worked<br>
<a href="#s5" style="color: #0b3d91; background: #ffffff;">4. How to find improvements</a> &mdash; the method, the pattern catalog, a <code style="background: #eef1f6; color: #111111; padding: 1px 5px; border-radius: 4px; font-size: 92%;">qccscope</code> demo<br>
<a href="#s6" style="color: #0b3d91; background: #ffffff;">5. Deep dives</a> &mdash; invsqrt, DQI, chemistry, with the exact simulator <code style="background: #eef1f6; color: #111111; padding: 1px 5px; border-radius: 4px; font-size: 92%;">kmsim</code><br>
<a href="#s7" style="color: #0b3d91; background: #ffffff;">6. The public lineage</a> &mdash; how the frontier was built, and by whom<br>
<a href="#s8" style="color: #0b3d91; background: #ffffff;">7. Open problems</a> &nbsp;&middot;&nbsp; <a href="#s9" style="color: #0b3d91; background: #ffffff;">8. Full credits and licences</a> &nbsp;&middot;&nbsp; <a href="#app" style="color: #0b3d91; background: #ffffff;">Appendix: code</a>
</div>

Code cells are hidden to keep the story readable; click **Code** on any cell to see it. Everything runs here: the
notebook rebuilds our submission from public inputs, verifies it with the official verifiers and draws every figure from
that run. If you find it useful, an upvote helps others find it, and corrections are very welcome in the comments.

<a id="ack"></a>
<div style="font-family: 'Segoe UI', Roboto, 'Helvetica Neue', Arial, sans-serif; background: #0b1f4b; color: #ffffff; padding: 10px 16px; border-radius: 8px; font-size: 22px; font-weight: 700;">0 &middot; Acknowledgements and who did what</div>

<div style="font-family: 'Segoe UI', Roboto, 'Helvetica Neue', Arial, sans-serif; font-size: 15.5px; line-height: 1.6; color: #111111; background: #ffffff;">
Almost every circuit in this notebook is other people's public work, and so is much of what we learned. Thank you to
<b>Syed Asad Ali</b> (the from-source invsqrt, DQI and chemistry builders), <b>Ali Haydar &Ouml;zda&#287;</b> (invsqrt <code style="background: #eef1f6; color: #111111; padding: 1px 5px; border-radius: 4px; font-size: 92%;">4560ac0c</code>),
<b>Areeb Ali</b> (product retention), <b>Ertu&#287;rul &Ouml;zer</b> (the DQI extractor cutoff, pinned caches and tables), <b>maoxin123</b>
(chemistry table, coded words and loaders; the 392k DQI decoder), <b>Atakan Aldemir</b> (weight update, readout windows,
shared products, the portfolios we read), <b>Heliosli</b> (chemistry weights and the public circuits dataset), <b>prvsiyan</b>
(refit code), <b>Lera (marculera)</b> and the authors her NOTICE credits, <b>Alex Chilton</b> (the verified-portfolio
practice), <b>WCao24</b>, <b>Thomas Gamet</b>, <b>manwithacat (James Barlow)</b> (the public-circuits dataset behind our
landscapes), <b>Kirill Tymchishin</b>, <b>SomeJoint</b>, <b>busyaprime</b>, <b>yoyobar</b>, <b>mrmeows</b>, <b>Hosen42</b>, <b>Gengsr</b>,
<b>Ahmed Berat &Ouml;zer</b>, the team <b>i want to run gemma 9b</b>, <b>jday96314</b> and <b>Oleg Sher</b>, whose
<i>Quantum Circuits from Zero</i> helped us read the invsqrt circuit. And Google Quantum AI and Kaggle for the
competition, the open kickmix simulator and the verifiers. Every change with its link is in <a href="#s9">section 8</a>.
If we missed you or misattributed something, please tell us in the comments and we will fix it.
</div>

<table style="font-family: 'Segoe UI', Roboto, 'Helvetica Neue', Arial, sans-serif; font-size: 14.5px; color: #111111; border-collapse: collapse; margin-top: 8px; background: #ffffff;">
<tr style="background: #e3ecf8;"><th style="background: #ffffff; color: #111111; text-align: left; padding: 6px 9px;">Task</th><th style="background: #ffffff; color: #111111; text-align: left; padding: 6px 9px;">File in the submission</th><th style="background: #ffffff; color: #111111; text-align: left; padding: 6px 9px;">Built by (latest change first)</th></tr>
<tr style="background: #ffffff; color: #111111;"><td style="background: #ffffff; color: #111111; padding: 6px 9px;"><b style="color: #1f6fd1;">invsqrt</b></td><td style="background: #ffffff; color: #111111; padding: 6px 9px;"><code style="background: #eef1f6; color: #111111; padding: 1px 5px; border-radius: 4px; font-size: 92%;">dcde7078</code> + <b>our dead-write fix</b> (46 qubits)</td><td style="background: #ffffff; color: #111111; text-align: left; padding: 6px 9px;"><b>asteriospanoras: one Toffoli replaced by a measurement and a free CZ</b> (78,765.51 &rarr; 78,719.53); the public 46-qubit line: <a href="https://www.kaggle.com/code/syed3000/vqcc-own-circuit-portfolio-notebook">Syed Asad Ali</a>'s own-circuit invsqrt and <a href="https://www.kaggle.com/code/ahoirg/qcc-submit-verify-and-package">Ali Haydar &Ouml;zda&#287;</a>'s <code style="background: #eef1f6; color: #111111; padding: 1px 5px; border-radius: 4px; font-size: 92%;">4560ac0c</code>, as shipped by <a href="https://www.kaggle.com/code/alexchilton/24-94-lb-verified-three-task-portfolio">Alex Chilton</a> and <a href="https://www.kaggle.com/code/atakanaldemir/qcc-gap-3-residue-repair-and-c43-native-compiler">Atakan Aldemir</a></td></tr>
<tr style="background: #f6f8fb;"><td style="background: #ffffff; color: #111111; padding: 6px 9px;"><b style="color: #d9541e;">DQI</b></td><td style="background: #ffffff; color: #111111; padding: 6px 9px;"><code style="background: #eef1f6; color: #111111; padding: 1px 5px; border-radius: 4px; font-size: 92%;">821bf8d1</code> (889 qubits)</td><td style="background: #ffffff; color: #111111; text-align: left; padding: 6px 9px;"><a href="https://www.kaggle.com/code/areebbb/qai-fewer-toffolis-same-qubits-lb-27-11042">Areeb Ali</a>'s 241 retained products on Syed Asad Ali's v12 decoder at extractor cutoff 18 (<a href="https://www.kaggle.com/code/ozertuu/quantum-ai-27-083-vqcc-dqi-cutoff-17">Ertu&#287;rul &Ouml;zer</a>); earlier decoder lineage credited in section 8</td></tr>
<tr style="background: #ffffff; color: #111111;"><td style="background: #ffffff; color: #111111; padding: 6px 9px;"><b style="color: #0f9a68;">chemistry</b></td><td style="background: #ffffff; color: #111111; padding: 6px 9px;">maoxin123's v27 table + loaders <code style="background: #eef1f6; color: #111111; padding: 1px 5px; border-radius: 4px; font-size: 92%;">71735b23</code>, <b>weights refitted by us</b> (427 qubits)</td><td style="background: #ffffff; color: #111111; text-align: left; padding: 6px 9px;"><a href="https://www.kaggle.com/code/maoxin123/qcc-v27-4033-toffoli-coded-word-chemistry">maoxin123</a>'s (8, 143, 44) table, coded words and v27 loaders; weights by <a href="https://www.kaggle.com/code/heliosli/the-dataset-is-public-now-enjoy">Heliosli</a> (and Atakan Aldemir) with prvsiyan's refit code; Lera (marculera)'s table line</td></tr>
<tr style="background: #f6f8fb;"><td style="background: #ffffff; color: #111111; padding: 6px 9px;"><b>this notebook</b></td><td style="background: #ffffff; color: #111111; padding: 6px 9px;">analysis</td><td style="background: #ffffff; color: #111111; text-align: left; padding: 6px 9px;">asteriospanoras: the experiments of section 3, <code style="background: #eef1f6; color: #111111; padding: 1px 5px; border-radius: 4px; font-size: 92%;">qccscope</code>, <code style="background: #eef1f6; color: #111111; padding: 1px 5px; border-radius: 4px; font-size: 92%;">kmsim</code>, the figures, and the <a href="https://www.kaggle.com/datasets/asteriospanoras/qcc-circuit-atlas">QCC Circuit Atlas</a>; landscapes from <a href="https://www.kaggle.com/datasets/manwithacat/vqcc-apache-public-circuits">manwithacat's public-circuits dataset</a></td></tr>
</table>

<div style="font-family: 'Segoe UI', Roboto, 'Helvetica Neue', Arial, sans-serif; font-size: 13.5px; color: #333333; margin-top: 4px; background: #ffffff;">The table describes the submission at the time of writing (2026-11-14). The code below rebuilds it from the attached public notebooks and prints the file hashes it actually used.</div>

### Setup: rebuild the submission and verify it (about 15 minutes)
<div style="font-family: 'Segoe UI', Roboto, 'Helvetica Neue', Arial, sans-serif; font-size: 15px; line-height: 1.55; color: #111111; background: #fff7e0; border-left: 6px solid #c98a00; padding: 11px 16px; border-radius: 8px;">
The next cell does exactly what our solution notebook
(<a href="https://www.kaggle.com/code/asteriospanoras/qcc-27-3122-frontier-finer-chemistry-refit">QCC 27.307</a>) does:
it takes maoxin123's public v27 files pinned by SHA-256 (the invsqrt file already carries our dead-write fix), puts in
our refitted chemistry weights, runs the three <b>unchanged official verifiers</b> on the exact bytes, and writes the same
<code style="background: #eef1f6; color: #111111; padding: 1px 5px; border-radius: 4px; font-size: 92%;">submission.zip</code> (same bytes). The zip goes to <code style="background: #eef1f6; color: #111111; padding: 1px 5px; border-radius: 4px; font-size: 92%;">/kaggle/working</code> only when
<code style="background: #eef1f6; color: #111111; padding: 1px 5px; border-radius: 4px; font-size: 92%;">WRITE_SUBMISSION = True</code> (first code cell). Every later figure reads this run's official receipt and the
circuits it built; all later cells are plain Python (numpy + matplotlib), no kickmix.
</div>

In [ ]:
# Runs on Kaggle (needs kickmix and the competition data). Same build as our solution notebook.
inp = pathlib.Path("/kaggle/input")
KM = next(p for p in inp.rglob("kickmix") if p.is_dir() and (p.parent / "verifiers").is_dir())
VER = KM.parent / "verifiers"; COMP = KM.parent; sys.path.insert(0, str(KM))
import kickmix as km
sha = lambda b: hashlib.sha256(b).hexdigest()
SUB = pathlib.Path("/tmp/sub"); (SUB / "dfthcsos").mkdir(parents=True, exist_ok=True)
TMP = pathlib.Path("/tmp/a"); TMP.mkdir(exist_ok=True)
import io, lzma, base64
import numpy as np
def _npz_sha(z):
    try:
        with zipfile.ZipFile(z) as Z: return sha(Z.read("dfthcsos/submission.npz"))
    except Exception: return ""
src = next(z for z in inp.rglob("submission.zip") if _npz_sha(z).startswith("3f864a7a"))  # maoxin123's v27 portfolio
PIN = {"invsqrt.kmb": "e46d90c3", "dqi.kmb": "821bf8d1", "dfthcsos/submission.npz": "3f864a7a", "dfthcsos/angle_k_0.kmb": "71735b23"}
with zipfile.ZipFile(src) as Z:
    for n in Z.namelist():
        if n.endswith("/"): continue
        b = Z.read(n)
        if n in PIN: assert sha(b).startswith(PIN[n]), (n, sha(b)[:12])
        (SUB / n).parent.mkdir(parents=True, exist_ok=True); (SUB / n).write_bytes(b)
log("public files", {n: sha((SUB / n).read_bytes())[:8] for n in PIN})
# Our chemistry change (same as our notebook "QCC 27.307"): W and H2_I refitted at fixed words; lambda_eff 46.907014 -> 46.906590.
REFIT_B64 = "REMOVED_EMBEDDED_BINARY"
new_w = np.load(io.BytesIO(lzma.decompress(base64.b64decode(REFIT_B64))))
assert sha(new_w["W"].tobytes() + new_w["H2_I"].tobytes()) == "f5eff7a73385adbde397546733245accde1e2e3547198ce9757aa7484643c6de"
_old = np.load(SUB / "dfthcsos/submission.npz"); arrays = {k: np.array(_old[k]) for k in _old.files}
arrays["W"] = np.array(new_w["W"]); arrays["H2_I"] = np.array(new_w["H2_I"])
with open(SUB / "dfthcsos/submission.npz", "wb") as f: np.savez(f, **arrays)
log("chemistry npz with our refitted weights", sha((SUB / "dfthcsos/submission.npz").read_bytes())[:12])
# invsqrt e46d90c3 already carries our dead-write fix (op 3292: CCX q16 q20 q21 -> HMR q21 b61; CZ q16 q20 if b61).
# For the analysis we also rebuild the circuit before the fix (CCX restored, measurement right after it).
c = km.Circuit.from_file(str(SUB / "invsqrt.kmb"), format="kmb"); c.to_file("/tmp/inv_fixed.kmx", format="kmx")
lines = open("/tmp/inv_fixed.kmx").read().splitlines()
k = lines.index("HMR q21 b61"); assert lines[k + 1] == "CZ q16 q20 if b61", lines[k:k + 2]
open("/tmp/inv.kmx", "w").write("\n".join(lines[:k] + ["CCX q16 q20 q21", "HMR q21 b61"] + lines[k + 2:]) + "\n")
log("invsqrt (fixed)", sha((SUB / "invsqrt.kmb").read_bytes())[:12])
def verify(args, timeout=4 * 3600):
    env = dict(os.environ, PYTHONPATH=f"{KM}:{VER}")
    r = subprocess.run([sys.executable, *map(str, args)], capture_output=True, text=True, env=env, cwd=str(VER), timeout=timeout)
    try: return json.loads(r.stdout[r.stdout.index("{"):])
    except Exception: return {"status": "ERROR", "stderr": r.stderr[-2000:]}
res = {}
res["dqi"] = verify([VER / "dqi/verify_submission_dqi.py", SUB / "dqi.kmb"])
log("dqi", {k: res["dqi"].get(k) for k in ("status", "score", "toffoli_count", "num_qubits", "value_errors", "phase_errors")})
res["dfthcsos"] = verify([VER / "dfthcsos/verify.py", SUB / "dfthcsos"])
log("chemistry", {k: res["dfthcsos"].get(k) for k in ("status", "score", "toffoli_count", "num_qubits", "lambda_eff")})
res["invsqrt"] = verify([VER / "invsqrt/verify_submission_invsqrt.py", SUB / "invsqrt.kmb"])
log("invsqrt", {k: res["invsqrt"].get(k) for k in ("status", "score", "num_qubits", "toffoli_count", "worst_error_over_tolerance")})
assert all(res[k].get("status") == "PASS" for k in res), res
OUT_ZIP = "/kaggle/working/submission.zip" if WRITE_SUBMISSION else "/tmp/submission.zip"
with zipfile.ZipFile(OUT_ZIP, "w", zipfile.ZIP_DEFLATED) as z:
    for p in sorted(SUB.rglob("*")):
        if p.is_file():
            zi = zipfile.ZipInfo(str(p.relative_to(SUB)), date_time=(2026, 11, 14, 0, 0, 0)); zi.compress_type = zipfile.ZIP_DEFLATED
            z.writestr(zi, p.read_bytes())
receipt = {"sha256": {str(p.relative_to(SUB)): sha(p.read_bytes()) for p in SUB.rglob("*") if p.is_file()}, "verify": res}
json.dump(receipt, open("/kaggle/working/receipt.json", "w"), indent=1, default=str)
log("submission.zip written to", OUT_ZIP, sha(open(OUT_ZIP, "rb").read())[:12])
DQI_SALTS = {}  # this notebook runs the default salt only
# Readable .kmx copies for the analysis (written outside SUB, so the zip above is unaffected).
def to_kmx(src_, dst):
    km.Circuit.from_file(str(src_), format=pathlib.Path(src_).suffix[1:]).to_file(str(dst), format="kmx"); return pathlib.Path(dst)
INV, INV_ORIG = pathlib.Path("/tmp/inv_fixed.kmx"), pathlib.Path("/tmp/inv.kmx"); NPZ = SUB / "dfthcsos" / "submission.npz"
BASE = to_kmx(COMP / "sample_submission" / "invsqrt.kmb", TMP / "invsqrt_reference.kmx") if (COMP / "sample_submission" / "invsqrt.kmb").exists() else None
CHEM_KMX = [to_kmx(p, TMP / f"{p.stem}.kmx") for p in sorted((SUB / "dfthcsos").glob("angle_k_*.kmb"), key=lambda p: int(p.stem.split("_")[-1]))]
PUB_KMX = {}
for p in sorted(inp.rglob("*.kmb")):
    if "/invsqrt/" in str(p) and p.stem[:12] not in PUB_KMX: PUB_KMX[p.stem[:12]] = to_kmx(p, TMP / f"pub_{p.stem[:12]}.kmx")
man = list(csv.DictReader(open(next(p for p in inp.rglob("manifest.csv") if "vqcc-apache" in str(p).lower() or "public-circuits" in str(p).lower()))))
log(len(PUB_KMX), "public invsqrt circuits;", len(man), "manifest records;", len(CHEM_KMX), "chemistry loaders")

In [ ]:
# The submission's official numbers (from this run's receipt) and the hosts' reference costs.
REF = {"invsqrt": 2076774.23, "dqi": 24439463583672.93, "dfthcsos": 3355599588.32}  # reference cost = score 1.0
REF_QT = {"invsqrt": (260, 7988.0), "dqi": (1893, 5725461.0), "dfthcsos": (730, 93237.0)}  # paper Sec. III; SCORING_README
R0 = 1.1912604648115832  # DQI repetition factor at zero failed shots out of 102,400 (verifier formula)
v = receipt["verify"]
rows = {  # task: (qubits, Toffolis per shot, cost; DQI priced at zero failed shots)
    "invsqrt": (v["invsqrt"]["num_qubits"], v["invsqrt"]["toffoli_count"], v["invsqrt"]["score"]),
    "dqi": (v["dqi"]["num_qubits"], v["dqi"]["toffoli_count"], v["dqi"]["score"] / v["dqi"]["repetition_factor"] * R0),
    "dfthcsos": (v["dfthcsos"]["num_qubits"], v["dfthcsos"]["toffoli_count"], v["dfthcsos"]["score"]),
}
def fnum(x):
    try: return float(x)
    except (TypeError, ValueError): return math.nan
ratios = {k: REF[k] / c for k, (q, t, c) in rows.items()}
gm = math.prod(ratios.values()) ** (1 / 3)
print("submission files:", {k: s[:12] for k, s in receipt["sha256"].items() if not k.startswith("dfthcsos/angle")})
print(f"{'task':9s} {'Q':>5s} {'Toffolis/shot':>14s} {'cost':>20s} {'ref/cost':>9s}")
for k, (q, t, c) in rows.items(): print(f"{k:9s} {q:5d} {t:14,.2f} {c:20,.2f} {ratios[k]:9.4f}")
print(f"geometric mean (DQI at zero failed shots): {gm:.5f}; DQI failed shots in this run: "
      f"{v['dqi']['value_errors'] + v['dqi']['phase_errors']}")
for salt, r_ in DQI_SALTS.items():
    print(f"DQI official run, salt {salt:8s}: {r_.get('status')}, {r_.get('num_qubits')} qubits, {r_.get('toffoli_count', math.nan):,.2f} Toffolis, value errors {r_.get('value_errors')}, phase errors {r_.get('phase_errors')}")

In [ ]:
"""kmsim: pure-Python/numpy kickmix-subset simulator for analysis (not the official verifier).

Semantics from kickmix op_type.h: HMR = Hadamard+measure+reset (outcome random m, phase (-1)^(m*v), qubit -> 0,
bit <- m); R = reset (error if qubit was 1); conditions: op runs only if all pushed condition bits are 1 and its own
`if b` bit is 1. Phase tracked in units of 2^-64 turns (uint64), Z_POW angles are half-turns.
"""
from fractions import Fraction
import numpy as np

MASK = (1 << 64) - 1
TOFF = {"CCX", "CCZ", "CSWAP"}


def half_turns_units(s):
    f = Fraction(s)  # exact decimal
    return int(round(f * (1 << 63))) & MASK  # half-turn = 2^63 units


def parse(path):
    ops = []
    regs = {}
    for line in open(path):
        t = line.split()
        if not t:
            continue
        if t[0] == "APPEND_TO_REGISTER":
            regs.setdefault(t[2], []).append(int(t[1][1:])); continue
        if t[0] == "REGISTER":
            continue
        cond = None
        if "if" in t:
            i = t.index("if"); cond = int(t[i + 1][1:]); t = t[:i]
        name = t[0]; args = t[1:]
        qs = [int(a[1:]) for a in args if a.startswith("q")]
        bs = [int(a[1:]) for a in args if a.startswith("b")]
        ang = None
        if name == "Z_POW":
            ang = half_turns_units(args[1])
        ops.append((name, qs, bs, cond, ang))
    return ops, regs


class Sim:
    def __init__(self, ops, nq, nb, batch, seed=0):
        self.ops = ops; self.B = batch
        self.q = np.zeros((nq, batch), dtype=bool)
        self.b = np.zeros((nb, batch), dtype=bool)
        self.ph = np.zeros(batch, dtype=np.uint64)
        self.rng = np.random.default_rng(seed)
        self.toff = np.zeros(batch, dtype=np.int64)
        self.rot = np.zeros(batch, dtype=np.int64)
        self.bad_reset = np.zeros(batch, dtype=bool)

    def run(self, start=0, stop=None, hook=None):
        stack = getattr(self, "stack", [])
        q, b = self.q, self.b
        ones = np.ones(self.B, dtype=bool)
        for k in range(start, len(self.ops) if stop is None else stop):
            name, qs, bs, cond, ang = self.ops[k]
            if name == "PUSH_CONDITION":
                stack.append(cond); continue
            if name == "POP_CONDITION":
                stack.pop(); continue
            act = ones
            if stack or cond is not None:
                act = ones.copy()
                for c in stack: act &= b[c]
                if cond is not None: act &= b[cond]
            if name == "CX":
                q[qs[1]] ^= q[qs[0]] & act
            elif name == "CZ":
                self._neg(q[qs[0]] & q[qs[1]] & act)
            elif name == "Z":
                self._neg(q[qs[0]] & act)
            elif name == "X":
                q[qs[0]] ^= act
            elif name == "CCX":
                q[qs[2]] ^= q[qs[0]] & q[qs[1]] & act; self.toff += act
            elif name == "CCZ":
                self._neg(q[qs[0]] & q[qs[1]] & q[qs[2]] & act); self.toff += act
            elif name == "SWAP":
                a, c = q[qs[0]].copy(), q[qs[1]].copy()
                q[qs[0]] = np.where(act, c, a); q[qs[1]] = np.where(act, a, c)
            elif name == "CSWAP":
                m = act & q[qs[0]]
                a, c = q[qs[1]].copy(), q[qs[2]].copy()
                q[qs[1]] = np.where(m, c, a); q[qs[2]] = np.where(m, a, c); self.toff += act
            elif name == "NEG":
                self._neg(act)
            elif name == "HMR":
                m = self.rng.random(self.B) < 0.5
                self._neg(m & q[qs[0]] & act)
                q[qs[0]] &= ~act
                b[bs[0]] = np.where(act, m, b[bs[0]])
            elif name == "R":
                self.bad_reset |= q[qs[0]] & act
                q[qs[0]] &= ~act
            elif name == "Z_POW":
                m = q[qs[0]] & act
                self.ph = self.ph + np.where(m, np.uint64(ang), np.uint64(0)); self.rot += act
            elif name == "BIT_STORE0":
                b[bs[0]] &= ~act
            elif name == "BIT_STORE1":
                b[bs[0]] |= act
            elif name == "BIT_INVERT":
                b[bs[0]] ^= act
            elif name in ("INFO_TAG", "DEBUG_PRINT"):
                pass
            else:
                raise ValueError(name)
            if hook is not None:
                hook(k, self)
        self.stack = stack

    def _neg(self, m):
        self.ph = self.ph + np.where(m, np.uint64(1 << 63), np.uint64(0))


def target_and_tol(xs, precision_bits=24):
    import math
    rs = [math.isqrt((1 << 130) // int(x)) for x in xs]
    target = np.array([((r + 1) >> 1) & MASK for r in rs], dtype=np.uint64)
    tol = np.array([min(r >> (precision_bits + 1), 1 << 63) for r in rs], dtype=np.uint64)
    return target, tol


def check(path, xs, seed=0, ops_regs=None):
    ops, regs = ops_regs or parse(path)
    nq = 1 + max(max(o[1]) for o in ops if o[1])
    nb = 1 + max((max(o[2]) for o in ops if o[2]), default=0)
    nb = max(nb, 1 + max((o[3] for o in ops if o[3] is not None), default=0))
    s = Sim(ops, nq, nb, len(xs), seed)
    xq = regs["r0"]
    for i, qi in enumerate(xq):
        s.q[qi] = (xs >> i) & 1 == 1
    s.run()
    xr = sum((s.q[qi].astype(np.uint64) << np.uint64(i)) for i, qi in enumerate(xq))
    restored = np.array_equal(xr, xs.astype(np.uint64))
    scratch = [i for i in range(nq) if i not in xq]
    dirty = s.q[scratch].any(axis=0)
    target, tol = target_and_tol(xs)
    err = (s.ph - target).view(np.int64)
    used = np.abs(err).astype(np.float64)
    return dict(nq=nq, restored=restored, dirty=int(dirty.sum()), bad_reset=int(s.bad_reset.sum()),
                worst_ratio=float((used / tol.astype(np.float64)).max()), toff_mean=float(s.toff.mean()),
                rot_mean=float(s.rot.mean())), s

In [ ]:
# Small quantum-circuit drawer and a runner for kmx snippets (used by the identities below).
def draw_circuit(ax, wires, ops, ncols, toff=C[1], title=None):
    """Draw a small circuit. wires: labels top to bottom ('b:...' = classical bit). ops: (col, NAME, *wires[, {'if': bit}])."""
    y = {w: -i for i, w in enumerate(wires)}
    for w in wires:
        if w.startswith("b:"):
            for d in (-0.035, 0.035): ax.plot([-0.4, ncols - 0.4], [y[w] + d] * 2, color=MUTED, lw=0.8, zorder=1)
        else: ax.plot([-0.4, ncols - 0.4], [y[w]] * 2, color=INK2, lw=1.1, zorder=1)
        ax.text(-0.55, y[w], w[2:] if w.startswith("b:") else w, ha="right", va="center", fontsize=11.5, color=INK2 if w.startswith("b:") else INK)
    dot = lambda x, w, col: ax.add_patch(Circle((x, y[w]), 0.085, color=col, zorder=4))
    def plus(x, w, col):
        ax.add_patch(Circle((x, y[w]), 0.19, facecolor=SURF, edgecolor=col, lw=1.4, zorder=4))
        ax.plot([x - 0.19, x + 0.19], [y[w]] * 2, color=col, lw=1.2, zorder=5); ax.plot([x, x], [y[w] - 0.19, y[w] + 0.19], color=col, lw=1.2, zorder=5)
    def cross(x, w, col):
        for s in (1, -1): ax.plot([x - 0.12, x + 0.12], [y[w] - 0.12 * s, y[w] + 0.12 * s], color=col, lw=1.6, zorder=5)
    def box(x, w, label, col=INK, fc=SURF, wd=0.62):
        ax.add_patch(FancyBboxPatch((x - wd / 2, y[w] - 0.24), wd, 0.48, boxstyle="round,pad=0.02", facecolor=fc, edgecolor=col, lw=1.2, zorder=4))
        ax.text(x, y[w], label, ha="center", va="center", fontsize=8.5, color=col, zorder=5, fontweight="bold")
    n_toff = 0
    for op in ops:
        kw = op[-1] if isinstance(op[-1], dict) else {}
        args = op[2:-1] if kw else op[2:]
        x, name = op[0], op[1]
        col = toff if name in ("CCX", "CCZ", "CSWAP") else INK
        n_toff += name in ("CCX", "CCZ", "CSWAP")
        ys = [y[a] for a in args]
        if len(ys) > 1: ax.plot([x, x], [min(ys), max(ys)], color=col, lw=1.4, zorder=3)
        if name in ("CX", "CCX"): [dot(x, a, col) for a in args[:-1]]; plus(x, args[-1], col)
        elif name in ("CZ", "CCZ"): [dot(x, a, col) for a in args]
        elif name == "CSWAP": dot(x, args[0], col); cross(x, args[1], col); cross(x, args[2], col)
        elif name == "SWAP": cross(x, args[0], col); cross(x, args[1], col)
        elif name == "X": plus(x, args[0], col)
        elif name in ("Z", "NOT", "R"): box(x, args[0], name, wd=0.42)
        elif name == "ZPOW": box(x, args[0], args[1], col=C[6], wd=0.9)
        elif name == "HMR":
            box(x, args[0], "HMR", col=C[0], wd=0.86); yb = y[args[1]]
            for d in (-0.03, 0.03): ax.plot([x + d, x + d], [y[args[0]] - 0.24, yb + 0.08], color=C[0], lw=0.8, zorder=3)
            ax.plot([x], [yb + 0.06], marker="v", color=C[0], ms=6, zorder=5)
        elif name == "BOX":
            lo, hi = min(ys), max(ys)
            ax.add_patch(FancyBboxPatch((x - 0.42, lo - 0.3), 0.84, hi - lo + 0.6, boxstyle="round,pad=0.02", facecolor="#eef4fc", edgecolor=C[0], lw=1.0, zorder=4))
            ax.text(x, (lo + hi) / 2, kw.get("label", ""), ha="center", va="center", fontsize=8.5, color=NAVY, zorder=5)
        if "if" in kw:
            yb, yg = y[kw["if"]], (min(ys) if y[kw["if"]] < min(ys) else max(ys))
            for d in (-0.03, 0.03): ax.plot([x + d, x + d], [yb, yg], color=INK2, lw=0.9, ls=(0, (3, 2)), zorder=2)
            ax.add_patch(Rectangle((x - 0.07, yb - 0.07), 0.14, 0.14, color=INK2, zorder=5))
    ax.set_xlim(-2.0, ncols - 0.2); ax.set_ylim(-len(wires) + 0.35, 0.75); ax.set_aspect("equal"); ax.axis("off")
    if title: ax.set_title(title, loc="left", fontsize=12, color=INK, pad=4)
    return n_toff

def run_text(txt, init, nq, nb, batch=64, seed=0):
    """Run a small kmx text circuit in kmsim from basis state `init` (list of bits per qubit) on `batch` shots."""
    ops_ = [ln for ln in txt.strip().splitlines()]
    path = TMP / "snippet.kmx"; path.write_text("\n".join(ops_) + "\n")
    o, _ = parse(path); s = Sim(o, nq, nb, batch, seed)
    for i, bit in enumerate(init): s.q[i] = bool(bit)
    s.run(); return s

<a id="s1"></a>
<div style="font-family: 'Segoe UI', Roboto, 'Helvetica Neue', Arial, sans-serif; background: #0b1f4b; color: #ffffff; padding: 10px 16px; border-radius: 8px; font-size: 22px; font-weight: 700;">1 &middot; The competition in five minutes</div>

You submit three **kickmix circuits**: circuits built only from reversible classical gates, phase gates and
measurement-based uncomputation. Such a circuit maps each input bit string to one output bit string times a phase, so a
classical computer can check it shot by shot, tracking one bit per qubit plus one phase. That is what makes the
competition *verifiable*: the hosts' verifiers simulate your circuit on many inputs and reject it on a single wrong
shot (DQI tolerates rare failures, at a price). No quantum background is needed; the whole game is to make three
fixed computations cheaper in **qubits** and **Toffoli gates**.

<table style="font-family: 'Segoe UI', Roboto, 'Helvetica Neue', Arial, sans-serif; font-size: 14.5px; color: #111111; border-collapse: collapse; background: #ffffff;">
<tr style="background: #e3ecf8;"><th style="background: #ffffff; color: #111111; text-align: left; padding: 5px 8px;">Task</th><th style="background: #ffffff; color: #111111; text-align: left; padding: 5px 8px;">Science</th><th style="background: #ffffff; color: #111111; text-align: left; padding: 5px 8px;">What the circuit computes</th><th style="background: #ffffff; color: #111111; text-align: left; padding: 5px 8px;">Cost (lower is better)</th><th style="background: #ffffff; color: #111111; text-align: left; padding: 5px 8px;">Hosts' reference</th></tr>
<tr style="background: #ffffff; color: #111111;"><td style="background: #ffffff; color: #111111; padding: 5px 8px;"><b style="color: #2a78d6;">invsqrt</b></td><td style="background: #ffffff; color: #111111; text-align: left; padding: 5px 8px;">fusion: Coulomb phase of every electron pair</td><td style="background: #ffffff; color: #111111; text-align: left; padding: 5px 8px;">|x&#10217;|0&#10217; &rarr; e<sup>2&pi;i/&radic;x</sup>|x&#10217;|0&#10217; for every x in [1, 3,133,452]</td><td style="background: #ffffff; color: #111111; text-align: left; padding: 5px 8px;">Q &times; C, C = Toffolis + T/4</td><td style="background: #ffffff; color: #111111; text-align: left; padding: 5px 8px;">260 qubits, C = 7,988</td></tr>
<tr style="background: #ffffff; color: #111111;"><td style="background: #ffffff; color: #111111; padding: 5px 8px;"><b style="color: #eb6834;">DQI</b></td><td style="background: #ffffff; color: #111111; text-align: left; padding: 5px 8px;">optimisation: Decoded Quantum Interferometry</td><td style="background: #ffffff; color: #111111; text-align: left; padding: 5px 8px;">Reed&ndash;Solomon RS(4095, 4025) syndrome decoder over GF(2<sup>12</sup>): from syndromes S0..S69, apply the sign (&minus;1)<sup>&Sigma; popcount(e<sub>j</sub> &amp; M<sub>j</sub>)</sup> and restore everything</td><td style="background: #ffffff; color: #111111; text-align: left; padding: 5px 8px;">R(&epsilon;<sub>+</sub>) &times; Q<sup>2</sup> &times; T</td><td style="background: #ffffff; color: #111111; text-align: left; padding: 5px 8px;">1,893 qubits, 5,725,461 Toffolis</td></tr>
<tr style="background: #ffffff; color: #111111;"><td style="background: #ffffff; color: #111111; padding: 5px 8px;"><b style="color: #1baf7a;">chemistry</b><br><code style="background: #eef1f6; color: #111111; padding: 1px 5px; border-radius: 4px; font-size: 92%;">dfthcsos</code></td><td style="background: #ffffff; color: #111111; text-align: left; padding: 5px 8px;">FeMoCo (nitrogen fixation): block encoding of the Hamiltonian</td><td style="background: #ffffff; color: #111111; text-align: left; padding: 5px 8px;">a family of QROM loaders |a&#10217;|z&#10217; &rarr; |a&#10217;|z &oplus; &delta;<sub>k</sub>(a)&#10217; that stream the Givens angles of a DF-THC factorization (also submitted)</td><td style="background: #ffffff; color: #111111; text-align: left; padding: 5px 8px;">&lambda;<sub>eff</sub> &times; T &times; Q over the whole block encoding</td><td style="background: #ffffff; color: #111111; text-align: left; padding: 5px 8px;">730 qubits, T = 93,237, &lambda;<sub>eff</sub> = 49.30</td></tr>
</table>

In [ ]:
# Figure: the three contracts at a glance (inputs, the circuit you write, what it must output, how it is tested).
fig, axs = plt.subplots(3, 1, figsize=(12, 8.3))
spec = [
    ("invsqrt", [r"$|x\rangle$  22 qubits", r"$|0\cdots0\rangle$  scratch"], "your circuit\n(phase by $1/\\sqrt{x}$)",
     [r"$e^{2\pi i/\sqrt{x}}\,|x\rangle$", r"$|0\cdots0\rangle$"],
     r"every $x$ in $[1,\ 3{,}133{,}452]$, 64 shots each (200.5M shots); phase error $\leq 2^{-24}/\sqrt{x}$ turns"),
    ("dqi", [r"$|S_0..S_{69}\rangle$  70 x 12 qubits", r"$E_0..E_{4094}$  classical record $M$", r"$|0\cdots0\rangle$  scratch"],
     "your decoder\n(Reed-Solomon over GF($2^{12}$))",
     [r"$(-1)^{\,e\cdot M}\,|S\rangle$  (S restored)", r"$M$  (classical)", r"$|0\cdots0\rangle$"],
     "102,400 random error patterns (up to 35 errors); rare failures allowed, priced by R($\\epsilon_+$); no Z_POW"),
    ("dfthcsos", [r"$|a\rangle$  selection (row address)", r"$|z\rangle$  J word registers", r"$|0\cdots0\rangle$  scratch"],
     "your loaders ANGLE$_k$\n(one QROM per batch)",
     [r"$|a\rangle$", r"$|z \oplus \delta_k(a)\rangle$", r"$|0\cdots0\rangle$"],
     "every address, 64 shots, forward and reverse; $\\leq$128 qubits per loader; the factorization sets $\\lambda_{eff}$"),
]
for ax, (task, ins, mid, outs, test) in zip(axs, spec):
    ax.set_xlim(0, 10); ax.set_ylim(-0.9, 2.2); ax.axis("off")
    col = TC[task]; n = len(ins); ys = np.linspace(1.75, 0.35, n) if n > 2 else [1.45, 0.65]
    ax.add_patch(FancyBboxPatch((3.85, 0.15), 2.3, 1.8, boxstyle="round,pad=0.04", facecolor=col, edgecolor=col, alpha=0.95))
    ax.text(5.0, 1.05, mid, ha="center", va="center", color="white", fontsize=11.5, fontweight="bold")
    for yy, s in zip(ys, ins):
        ax.text(3.55, yy, s, ha="right", va="center", fontsize=11.5, color=INK); ax.annotate("", (3.85, yy), (3.6, yy), arrowprops=dict(arrowstyle="-|>", color=INK2, lw=1))
    for yy, s in zip(ys, outs):
        ax.text(6.45, yy, s, ha="left", va="center", fontsize=11.5, color=INK); ax.annotate("", (6.42, yy), (6.15, yy), arrowprops=dict(arrowstyle="-|>", color=INK2, lw=1))
    ax.text(0.0, 2.1, TN[task], fontsize=13, fontweight="bold", color=col, va="center")
    ax.text(5.0, -0.45, "verifier: " + test, ha="center", va="center", fontsize=10.5, color=INK2)
headline(fig, "Three fixed computations: you choose the circuit, the verifier checks every shot",
         "inputs on the left, the contract on the right; scratch qubits must always come back to |0>")
plt.show()

### The score
Each task's cost is divided into the hosts' reference cost, and the three ratios are combined with a geometric mean:

$$\text{Score} = \left(\frac{\text{cost}^{\text{ref}}_{\text{invsqrt}}}{\text{cost}_{\text{invsqrt}}}\cdot\frac{\text{cost}^{\text{ref}}_{\text{dqi}}}{\text{cost}_{\text{dqi}}}\cdot\frac{\text{cost}^{\text{ref}}_{\text{chem}}}{\text{cost}_{\text{chem}}}\right)^{1/3}$$

$$\text{cost}_{\text{invsqrt}} = Q\times\Big(\text{Toffolis}+\tfrac{T}{4}\Big),\qquad \text{cost}_{\text{dqi}} = R(\epsilon_+)\,Q^2\,T,\qquad \text{cost}_{\text{chem}} = \lambda_{\text{eff}}\,T\,Q$$

Two consequences drive everything. **A 1% cut in any task's cost is worth the same** (one third of a percent of the
score), whatever the task. And because each cost is a *product*, a 1% cut in any factor (qubits, Toffolis, the
repetition factor, the chemistry normalisation) is worth the same 1% of that task, except DQI qubits, which count
twice (Q squared). Section 2 turns this into exchange rates.

<div style="font-family: 'Segoe UI', Roboto, 'Helvetica Neue', Arial, sans-serif; font-size: 15px; color: #111111; background: #fff6e8; border-left: 5px solid #eda100; padding: 10px 14px; border-radius: 6px; margin: 6px 0;">
<b>How a verifier tests a circuit.</b> Each shot writes an input into the registers, runs the circuit in kickmix and
checks three things: the inputs are restored, every scratch qubit is back to 0, and the tracked phase is right.
Inputs and the random outcomes of <code style="background: #eef1f6; color: #111111; padding: 1px 5px; border-radius: 4px; font-size: 92%;">HMR</code> measurements are seeded from the SHA-256 of your file plus a
<b>secret salt</b> on the leaderboard (an empty salt locally), so any change to a circuit gets fresh tests and the
leaderboard draws differ from yours. invsqrt runs every input 64 times; DQI draws 102,400 random error patterns and
turns its failures into an upper confidence bound &epsilon;<sub>+</sub> (one-sided Clopper&ndash;Pearson at
&delta; = 2<sup>&minus;128</sup>) and a repetition factor R(&epsilon;<sub>+</sub>) = ln 2 / &minus;ln(1/2 + 2&radic;&epsilon;<sub>+</sub>);
chemistry runs every address 64 times, forward then in reverse, and rejects the submission on any failure or if
the encoded Hamiltonian leaves the accuracy gates (20 mHa, &Lambda; &gt; 49.0, E<sub>gap</sub> &gt; 3.914898 Ha).
</div>

The four panels below show what each test means in numbers.

In [ ]:
# Figure: what the verifiers check, in numbers. The DQI functions follow the formulas in verify_submission_dqi.py.
def cp_upper(k, n=102400, log2_delta=-128.0):
    """One-sided Clopper-Pearson upper bound at confidence 1 - 2^log2_delta (as in the DQI verifier)."""
    ld = log2_delta * math.log(2.0)
    if k == 0: return -math.expm1(ld / n)
    j = np.arange(k + 1, dtype=float); lc = np.r_[0.0, np.cumsum(np.log(n - j[1:] + 1.0) - np.log(j[1:]))]
    def tail(p):
        t = lc + j * math.log(p) + (n - j) * math.log1p(-p); m = t.max(); return m + math.log(np.exp(t - m).sum())
    lo, hi = k / n, 1.0
    for _ in range(80):
        mid = 0.5 * (lo + hi); lo, hi = (mid, hi) if tail(mid) > ld else (lo, mid)
    return hi
def rep_factor(eps): return 1.0 if eps == 0 else math.log(2.0) / -math.log(0.5 + 2.0 * math.sqrt(eps))
def dqi_weights(m=4095, l=35, q=4096, r=2016):
    k = np.arange(l + 1, dtype=float); off = np.sqrt(k[1:] * (m - k[1:] + 1)); d = (q - 2 * r) / math.sqrt(r * (q - r))
    w = np.abs(np.linalg.eigh(np.diag(k * d) + np.diag(off, 1) + np.diag(off, -1))[1][:, -1]); return (w / np.linalg.norm(w)) ** 2
assert abs(rep_factor(cp_upper(0)) - R0) < 1e-12
fig, axs = plt.subplots(2, 2, figsize=(12, 7.6))
# (a) invsqrt target and tolerance
xg = np.geomspace(1, 3133452, 400); ax = axs[0, 0]
ax.plot(xg, 1 / np.sqrt(xg), color=TC["invsqrt"], label=r"target phase $1/\sqrt{x}$ (turns)")
ax.plot(xg, 2.0 ** -24 / np.sqrt(xg), color=C[3], label=r"tolerance $2^{-24}/\sqrt{x}$")
ax.fill_between(xg, 1e-12, 2.0 ** -24 / np.sqrt(xg), color=C[3], alpha=0.12)
ax.set_xscale("log"); ax.set_yscale("log"); ax.set_ylim(1e-11, 3); ax.set_xlabel("input x"); ax.set_ylabel("turns")
wx = v["invsqrt"]["worst_x"]; ax.scatter([wx], [2.0 ** -24 / math.sqrt(wx)], s=50, color=INK, zorder=4)
ax.annotate(f"portfolio's tightest input x = {wx}:\n{100 * v['invsqrt']['worst_error_over_tolerance']:.2f}% of the tolerance used", (wx, 2.0 ** -24 / math.sqrt(wx)), (60, 1e-6), fontsize=10, color=INK2, arrowprops=dict(arrowstyle="-", color=INK2))
ax.legend(loc="upper right", fontsize=10); ax.set_title("invsqrt: ~7 significant digits at all 3.1M inputs", loc="left")
# (b) DQI error-weight distribution
p = dqi_weights(); ax = axs[0, 1]; kk = np.arange(len(p))
ax.bar(kk, p, color=TC["dqi"], width=0.8); ax.set_xlabel("number of errors in a test shot"); ax.set_ylabel("probability")
ax.axvline((p * kk).sum(), color=INK2, lw=1, ls="--"); ax.text((p * kk).sum(), p.max() * 1.02, f" mean {(p * kk).sum():.2f}", color=INK2, fontsize=10, ha="right", va="bottom")
ax.set_title("DQI: most shots carry 29-34 errors (max 35)", loc="left"); ax.set_ylim(0, p.max() * 1.15); ax.set_xlim(14.5, 36)
# (c) DQI repetition factor by failed shots
ax = axs[1, 0]; ks = np.arange(6); Rk = np.array([rep_factor(cp_upper(int(k))) for k in ks])
extra = 100 * (Rk / Rk[0] - 1)
ax.bar([str(k) for k in ks], extra, color=TC["dqi"], width=0.6); ax.set_ylim(0, extra.max() * 1.3)
for k, (r, e) in enumerate(zip(Rk, extra)): ax.text(k, e, f"R = {r:.4f}\n{e:+.2f}%" if k else f"R = {r:.4f}\n(zero failures)", ha="center", va="bottom", fontsize=9.5, color=INK)
ax.set_xlabel("failed shots out of 102,400"); ax.set_ylabel("DQI cost above zero failures (%)")
ax.set_title("DQI: R = 1.19 at best; each failure adds ~0.4%", loc="left")
# (d) chemistry: lambda_eff at the E_gap floor
ax = axs[1, 1]; mt = v["dfthcsos"].get("metrics", {})
Lg = np.linspace(200, 330, 200); EG = 3.914898
ax.plot(Lg, np.sqrt(EG * (2 * Lg - EG)), color=TC["dfthcsos"], label=None); ax.text(236, 40.0, r"curve: $E_{gap}$ at its floor 3.914898 Ha", fontsize=10, color=INK2)
pts = [("hosts' baseline", 308.4559397330944, 3.965477244291833, MUTED)]
if "Lambda" in mt: pts.append(("portfolio", mt["Lambda"], mt["E_gap"], INK))
for lab, L_, E_, col in pts:
    le = math.sqrt(E_ * (2 * L_ - E_)); ax.scatter([L_], [le], s=55, color=col, zorder=4)
    ax.annotate(f"{lab}: Lambda {L_:.1f}\nE_gap {E_:.9f}\nlambda_eff {le:.3f}", (L_, le), (285, 42.3) if lab == "portfolio" else (205, 49.6), fontsize=9.5, color=INK2, va="center", arrowprops=dict(arrowstyle="-", color=INK2))
ax.set_xlabel(r"one-norm $\Lambda$ of the factorization (Ha)"); ax.set_ylabel(r"$\lambda_{eff}$ (Ha)")
ax.set_title(r"chemistry: smaller $\Lambda$, smaller $\lambda_{eff}$", loc="left")
headline(fig, "What the three verifiers ask for, in numbers", "computed from the verifiers' formulas; portfolio values from the official receipt")
plt.show()
if "E_gap" in mt:
    print(f"chemistry gates of the portfolio: error_1 {1000 * mt['error_1']:.6f} mHa, error_2 {1000 * mt['error_2']:.6f} mHa (limit 20), "
          f"E_gap {mt['E_gap']:.9f} Ha (floor 3.914898, strict), Lambda {mt['Lambda']:.3f} (floor 49.0)")
print("""Reading: every chemistry gate is pressed against its limit: the refits push the errors to just under 20 mHa and the
gap to just above its floor, because lambda_eff grows with E_gap. The score then rewards a smaller one-norm Lambda.""")

### The kickmix instruction set, on one card
A circuit is a text file (`.kmx`) or its binary form (`.kmb`), one instruction per line, for example `CCX q4 q5 q51`
or `CZ q0 q1 if b3`. Qubits are `q`, classical bits are `b`. Only three instructions cost anything.

<table style="font-family: 'Segoe UI', Roboto, 'Helvetica Neue', Arial, sans-serif; font-size: 14.5px; color: #111111; border-collapse: collapse; background: #ffffff;">
<tr style="background: #e3ecf8;"><th style="background: #ffffff; color: #111111; text-align: left; padding: 5px 8px;">Class</th><th style="background: #ffffff; color: #111111; text-align: left; padding: 5px 8px;">Instruction</th><th style="background: #ffffff; color: #111111; text-align: left; padding: 5px 8px;">Effect on a basis state</th><th style="background: #ffffff; color: #111111; text-align: left; padding: 5px 8px;">Price in the score</th></tr>
<tr style="background: #ffffff; color: #111111;"><td style="background: #ffffff; color: #111111; text-align: left; padding: 5px 8px;">reversible classical</td><td style="background: #ffffff; color: #111111; text-align: left; padding: 5px 8px;"><code style="background: #eef1f6; color: #111111; padding: 1px 5px; border-radius: 4px; font-size: 92%;">X</code>, <code style="background: #eef1f6; color: #111111; padding: 1px 5px; border-radius: 4px; font-size: 92%;">CX</code>, <code style="background: #eef1f6; color: #111111; padding: 1px 5px; border-radius: 4px; font-size: 92%;">SWAP</code></td><td style="background: #ffffff; color: #111111; text-align: left; padding: 5px 8px;">flip / XOR / exchange bits</td><td style="background: #ffffff; color: #111111; text-align: left; padding: 5px 8px;">free</td></tr>
<tr style="background: #ffffff; color: #111111;"><td style="background: #ffffff; color: #111111; text-align: left; padding: 5px 8px;"></td><td style="background: #ffffff; color: #111111; text-align: left; padding: 5px 8px;"><code style="background: #eef1f6; color: #111111; padding: 1px 5px; border-radius: 4px; font-size: 92%;">CCX</code> (Toffoli), <code style="background: #eef1f6; color: #111111; padding: 1px 5px; border-radius: 4px; font-size: 92%;">CSWAP</code> (Fredkin)</td><td style="background: #ffffff; color: #111111; text-align: left; padding: 5px 8px;">t &oplus;= a&middot;b; swap a, b when c = 1</td><td style="background: #ffffff; color: #111111; text-align: left; padding: 5px 8px;"><span style="background: #eb6834; color: #fff; border-radius: 8px; padding: 1px 7px;">1 Toffoli</span> each time it runs</td></tr>
<tr style="background: #ffffff; color: #111111;"><td style="background: #ffffff; color: #111111; text-align: left; padding: 5px 8px;">diagonal phase</td><td style="background: #ffffff; color: #111111; text-align: left; padding: 5px 8px;"><code style="background: #eef1f6; color: #111111; padding: 1px 5px; border-radius: 4px; font-size: 92%;">Z</code>, <code style="background: #eef1f6; color: #111111; padding: 1px 5px; border-radius: 4px; font-size: 92%;">CZ</code>, <code style="background: #eef1f6; color: #111111; padding: 1px 5px; border-radius: 4px; font-size: 92%;">NEG</code></td><td style="background: #ffffff; color: #111111; text-align: left; padding: 5px 8px;">multiply the phase by &minus;1 when all named qubits are 1 (NEG: always)</td><td style="background: #ffffff; color: #111111; text-align: left; padding: 5px 8px;">free</td></tr>
<tr style="background: #ffffff; color: #111111;"><td style="background: #ffffff; color: #111111; text-align: left; padding: 5px 8px;"></td><td style="background: #ffffff; color: #111111; text-align: left; padding: 5px 8px;"><code style="background: #eef1f6; color: #111111; padding: 1px 5px; border-radius: 4px; font-size: 92%;">CCZ</code></td><td style="background: #ffffff; color: #111111; text-align: left; padding: 5px 8px;">&minus;1 when a = b = c = 1</td><td style="background: #ffffff; color: #111111; text-align: left; padding: 5px 8px;"><span style="background: #eb6834; color: #fff; border-radius: 8px; padding: 1px 7px;">1 Toffoli</span></td></tr>
<tr style="background: #ffffff; color: #111111;"><td style="background: #ffffff; color: #111111; text-align: left; padding: 5px 8px;"></td><td style="background: #ffffff; color: #111111; text-align: left; padding: 5px 8px;"><code style="background: #eef1f6; color: #111111; padding: 1px 5px; border-radius: 4px; font-size: 92%;">Z_POW q t</code></td><td style="background: #ffffff; color: #111111; text-align: left; padding: 5px 8px;">phase e<sup>i&pi;t</sup> when q = 1 (t in half-turns)</td><td style="background: #ffffff; color: #111111; text-align: left; padding: 5px 8px;">invsqrt only: free if t is a multiple of 1/2; 1 T (= &frac14; Toffoli) if a multiple of 1/4; otherwise a <b>rotation</b> of 0.56&nbsp;log<sub>2</sub>(1/&epsilon;)&nbsp;+&nbsp;5.3 T. Forbidden in DQI and chemistry</td></tr>
<tr style="background: #ffffff; color: #111111;"><td style="background: #ffffff; color: #111111; text-align: left; padding: 5px 8px;">measurement</td><td style="background: #ffffff; color: #111111; text-align: left; padding: 5px 8px;"><code style="background: #eef1f6; color: #111111; padding: 1px 5px; border-radius: 4px; font-size: 92%;">HMR q b</code> (&ldquo;hammer&rdquo;)</td><td style="background: #ffffff; color: #111111; text-align: left; padding: 5px 8px;">X-basis measurement: random bit m into b, phase (&minus;1)<sup>m&middot;q</sup>, q reset to 0</td><td style="background: #ffffff; color: #111111; text-align: left; padding: 5px 8px;">free</td></tr>
<tr style="background: #ffffff; color: #111111;"><td style="background: #ffffff; color: #111111; text-align: left; padding: 5px 8px;"></td><td style="background: #ffffff; color: #111111; text-align: left; padding: 5px 8px;"><code style="background: #eef1f6; color: #111111; padding: 1px 5px; border-radius: 4px; font-size: 92%;">R q</code></td><td style="background: #ffffff; color: #111111; text-align: left; padding: 5px 8px;">reset a qubit that is already 0 (if it was 1, the shot fails)</td><td style="background: #ffffff; color: #111111; text-align: left; padding: 5px 8px;">free</td></tr>
<tr style="background: #ffffff; color: #111111;"><td style="background: #ffffff; color: #111111; text-align: left; padding: 5px 8px;">classical bits</td><td style="background: #ffffff; color: #111111; text-align: left; padding: 5px 8px;"><code style="background: #eef1f6; color: #111111; padding: 1px 5px; border-radius: 4px; font-size: 92%;">BIT_STORE0/1</code>, <code style="background: #eef1f6; color: #111111; padding: 1px 5px; border-radius: 4px; font-size: 92%;">BIT_INVERT</code></td><td style="background: #ffffff; color: #111111; text-align: left; padding: 5px 8px;">set or flip a classical bit</td><td style="background: #ffffff; color: #111111; text-align: left; padding: 5px 8px;">free</td></tr>
<tr style="background: #ffffff; color: #111111;"><td style="background: #ffffff; color: #111111; text-align: left; padding: 5px 8px;">control</td><td style="background: #ffffff; color: #111111; text-align: left; padding: 5px 8px;"><code style="background: #eef1f6; color: #111111; padding: 1px 5px; border-radius: 4px; font-size: 92%;">... if b</code>, <code style="background: #eef1f6; color: #111111; padding: 1px 5px; border-radius: 4px; font-size: 92%;">PUSH_CONDITION b</code> / <code style="background: #eef1f6; color: #111111; padding: 1px 5px; border-radius: 4px; font-size: 92%;">POP_CONDITION</code></td><td style="background: #ffffff; color: #111111; text-align: left; padding: 5px 8px;">run only when the bit(s) are 1</td><td style="background: #ffffff; color: #111111; text-align: left; padding: 5px 8px;">a conditioned Toffoli is counted only on the shots where it runs: the cost is a <b>mean per shot</b></td></tr>
</table>

### The one trick everyone uses: measurement-based uncomputation
A scratch qubit <code style="background: #eef1f6; color: #111111; padding: 1px 5px; border-radius: 4px; font-size: 92%;">t</code> that holds <code style="background: #eef1f6; color: #111111; padding: 1px 5px; border-radius: 4px; font-size: 92%;">a AND b</code> must be cleaned. Running the Toffoli again costs a second Toffoli. Instead,
<code style="background: #eef1f6; color: #111111; padding: 1px 5px; border-radius: 4px; font-size: 92%;">HMR t</code> measures it in the X basis: <code style="background: #eef1f6; color: #111111; padding: 1px 5px; border-radius: 4px; font-size: 92%;">t</code> is reset for free, but the state picks up the sign (&minus;1)<sup>m&middot;ab</sup>
for a random bit <code style="background: #eef1f6; color: #111111; padding: 1px 5px; border-radius: 4px; font-size: 92%;">m</code>. A free <code style="background: #eef1f6; color: #111111; padding: 1px 5px; border-radius: 4px; font-size: 92%;">CZ a b if m</code> removes exactly that sign. Half the Toffolis, same result; the verifier sees
a wrong sign on half the shots if the fix-up is missing. The figure draws both versions, and <code style="background: #eef1f6; color: #111111; padding: 1px 5px; border-radius: 4px; font-size: 92%;">kmsim</code> checks them on
every input and both measurement outcomes.

In [ ]:
# Figure + proof: compute-copy-uncompute with 2 Toffolis versus measurement-based uncomputation with 1 Toffoli.
fig, axs = plt.subplots(1, 3, figsize=(12.5, 3.8), gridspec_kw={"width_ratios": [4.8, 5.8, 5.8]})
W = ["a", "b", "t", "u", "b:m"]
n1 = draw_circuit(axs[0], W[:4], [(0, "CCX", "a", "b", "t"), (1, "CX", "t", "u"), (2, "CCX", "a", "b", "t")], 3)
n2 = draw_circuit(axs[1], W, [(0, "CCX", "a", "b", "t"), (1, "CX", "t", "u"), (2, "HMR", "t", "b:m"), (3, "CZ", "a", "b", {"if": "b:m"})], 4)
n3 = draw_circuit(axs[2], W, [(0, "CCX", "a", "b", "t"), (1, "CX", "t", "u"), (2, "HMR", "t", "b:m")], 4)
axs[0].set_title(f"uncompute by recomputing: {n1} Toffolis", loc="left", fontsize=11.5)
axs[1].set_title(f"measurement-based: {n2} Toffoli", loc="left", fontsize=11.5)
axs[2].set_title("fix-up forgotten: wrong sign", loc="left", fontsize=11.5, color=C[7])
headline(fig, "Measurement-based uncomputation halves the cost of a temporary AND",
         "all three copy a AND b into u and must leave t = 0; orange = Toffoli; the dashed line is a classical condition on bit m")
plt.show()
txt = {"recompute": "CCX q0 q1 q2\nCX q2 q3\nCCX q0 q1 q2", "measurement-based": "CCX q0 q1 q2\nCX q2 q3\nHMR q2 b0\nCZ q0 q1 if b0",
       "fix-up forgotten": "CCX q0 q1 q2\nCX q2 q3\nHMR q2 b0"}
for name, t_ in txt.items():
    bad = toffs = 0
    for a in (0, 1):
        for b in (0, 1):
            s = run_text(t_, [a, b, 0, 0], 4, 1, batch=256, seed=a * 2 + b)
            ok = (~s.q[2]) & (s.q[3] == bool(a & b)) & (s.ph == 0)
            bad += int((~ok).sum()); toffs = float(s.toff.mean())
    print(f"{name:18s}: Toffolis {toffs:.0f}, failed shots {bad} of 1,024 (4 inputs x 256 random outcomes)")

<a id="s2"></a>
<div style="font-family: 'Segoe UI', Roboto, 'Helvetica Neue', Arial, sans-serif; background: #0b1f4b; color: #ffffff; padding: 10px 16px; border-radius: 8px; font-size: 22px; font-weight: 700;">2 &middot; The score, the cost anatomy and the exchange rates</div>

The hosts' reference circuits are honest textbook constructions. The public frontier is now 11 to 68 times cheaper
per task. Because each cost is a product, the improvement of a task is the product of the improvements of its factors:
on a log scale they simply add up, which is how the first figure draws them.

In [ ]:
# Figure: each task's improvement over the reference, split into its multiplicative factors (log scale).
fac = {"invsqrt": [("qubits Q", REF_QT["invsqrt"][0] / rows["invsqrt"][0]), ("Toffoli-eq. C", REF_QT["invsqrt"][1] / v["invsqrt"]["toffoli_equivalent"])],
       "dqi": [("Q squared", (REF_QT["dqi"][0] / rows["dqi"][0]) ** 2), ("Toffolis T", REF_QT["dqi"][1] / rows["dqi"][1])],
       "dfthcsos": [("qubits Q", REF_QT["dfthcsos"][0] / rows["dfthcsos"][0]), ("Toffolis T", REF_QT["dfthcsos"][1] / rows["dfthcsos"][1]),
                    ("lambda_eff", 49.30136925057438 / v["dfthcsos"]["lambda_eff"])]}
fig, ax = plt.subplots(figsize=(12, 3.9))
shades = [1.0, 0.62, 0.35]
for i, task in enumerate(["invsqrt", "dqi", "dfthcsos"]):
    x0 = 1.0
    for j, (lab, f) in enumerate(fac[task]):
        ax.barh(i, x0 * f - x0, left=x0, color=TC[task], alpha=shades[j], height=0.56, edgecolor=SURF, linewidth=2)
        xm = math.sqrt(x0 * x0 * f)
        if f > 1.3: ax.text(xm, i, f"{lab}\nx{f:.2f}", ha="center", va="center", fontsize=10, color="white" if j == 0 else INK)
        else: ax.text(xm, i + 0.31, f"{lab} x{f:.3f}", ha="center", va="top", fontsize=9.5, color=INK2)
        x0 *= f
    ax.text(x0 * 1.06, i, f"{ratios[task]:.2f}x", va="center", fontsize=12.5, fontweight="bold", color=INK)
ax.set_axisbelow(True); ax.set_xscale("log"); ax.set_yticks(range(3)); ax.set_yticklabels([TN[t] for t in ["invsqrt", "dqi", "dfthcsos"]]); ax.invert_yaxis()
ax.set_xlim(1, 130); ax.xaxis.set_major_locator(FixedLocator([1, 2, 5, 10, 20, 50, 100])); ax.xaxis.set_major_formatter(FormatStrFormatter("%gx"))
ax.xaxis.set_minor_locator(FixedLocator([])); ax.set_xlabel("reference cost / portfolio cost (log scale: the factors add up)"); ax.grid(axis="y", visible=False)
headline(fig, f"The portfolio is {min(ratios.values()):.0f}-{max(ratios.values()):.0f}x cheaper than the reference per task: score {gm:.3f}",
         "score = geometric mean of the three bars; DQI priced at zero failed shots (R equal to the reference's)")
plt.show()

### Cost anatomy: where the cost of the portfolio goes
Four bars, one per cost factor that has parts. invsqrt pays for its rotations in Toffoli-equivalents; DQI's Toffolis
are mostly plain `CCX`; and in chemistry **almost half of the Toffolis and most of the qubits are priced by the
verifier's formula** for the state preparations and Givens rotations, not by the submitted loaders.

In [ ]:
# Figure: cost anatomy from the official receipt.
iv, dq, ch = v["invsqrt"], v["dqi"], v["dfthcsos"]; rs = ch["resources"]
rot_T = iv["toffoli_equivalent"] - iv["toffoli_count"] - iv.get("t_count", 0) / 4
socs = dq.get("shot_operation_counts", {}); nsh = dq["verified_shots"]
bars = [
    ("invsqrt: Toffoli-equivalents per shot", TC["invsqrt"], [("Toffolis", iv["toffoli_count"]), (f"{iv['rotation_count']:.0f} rotations", rot_T), ("T gates", iv.get("t_count", 0) / 4)]),
    ("DQI: Toffolis per shot, by gate", TC["dqi"], [(g, socs[g] / nsh) for g in ("CCX", "CSWAP", "CCZ") if g in socs] or [("Toffolis", dq["toffoli_count"])]),
    ("chemistry: Toffolis of one block encoding", TC["dfthcsos"], [("loaders, 2 passes (submitted)", rs["angles_toffoli"]), ("inner state prep (formula)", rs["inner_state_prep_toffoli"]),
                                                     ("Givens rotations (formula)", rs["givens_rotations_toffoli"]), ("outer prep", rs["outer_state_prep_toffoli"])]),
    ("chemistry: qubits", TC["dfthcsos"], [("shared temporary workspace (formula)", rs["temporary_qubits"]), ("angle words J x brot", rs["angles_persistent_qubits"]),
                                           ("inner prep", rs["inner_state_prep_persistent_qubits"]), ("outer prep", rs["outer_state_prep_persistent_qubits"])]),
]
fig, axs = plt.subplots(4, 1, figsize=(12, 5.8))
for ax, (lab, col, parts) in zip(axs, bars):
    tot = sum(p_ for _, p_ in parts); x0 = 0; small = []
    for j, (pl, val) in enumerate(parts):
        ax.barh(0, val, left=x0, color=col, alpha=[1.0, 0.62, 0.38, 0.2][j], height=0.6, edgecolor=SURF, linewidth=2)
        if val / tot > 0.12: ax.text(x0 + val / 2, 0, f"{pl}\n{val:,.0f} ({100 * val / tot:.0f}%)", ha="center", va="center", fontsize=9.5, color="white" if j == 0 else INK)
        else: small.append(f"{pl} {val:,.2f}" if val < 10 else f"{pl} {val:,.0f}")
        x0 += val
    if small: ax.text(tot, 0.36, "  |  ".join(small), ha="right", va="bottom", fontsize=9.5, color=INK2)
    ax.set_xlim(0, tot); ax.set_ylim(-0.4, 0.75); ax.axis("off"); ax.set_title(f"{lab}: {tot:,.2f}" if tot < 1e4 and tot != int(tot) else f"{lab}: {tot:,.0f}", loc="left", fontsize=11.5)
headline(fig, f"Where the cost goes: the formula carries {100 * rs['heuristic_toffoli'] / ch['toffoli_count']:.0f}% of chemistry's Toffolis",
         f"official receipt; 'formula' = priced by the verifier's model from (R, B, C) and brot; invsqrt rotations = {100 * rot_T / iv['toffoli_equivalent']:.0f}% of its C")
plt.show()
print(f"invsqrt rotations: {iv['rotation_count']:.0f} at eps = {iv['rotation_eps']:.3e}, "
      f"{0.56 * math.log2(1 / iv['rotation_eps']) + 5.3:.2f} T each = {(0.56 * math.log2(1 / iv['rotation_eps']) + 5.3) / 4:.2f} Toffoli-eq.")
print(f"chemistry: T_total = {rs['angles_toffoli']:,.0f} (loaders) + {rs['heuristic_toffoli']:,} (formula); "
      f"Q_total = {rs['persistent_qubits']} persistent + {rs['temporary_qubits']} shared temporary")

### Exchange rates: what one qubit, one Toffoli or one rotation is worth
The first thing to check before any circuit surgery. Removing one unit of a resource cuts that task's cost by a known
fraction; the geometric mean turns it into score points. The dots are drawn on a log scale; the labels also give the
same change in **Toffolis of that task**, the common currency inside a task.

In [ ]:
# Figure + table: exchange rates at the portfolio's operating point.
inv_C, inv_Q = iv["toffoli_equivalent"], iv["num_qubits"]; dqi_T, dqi_Q = dq["toffoli_count"], dq["num_qubits"]
ch_T, ch_Q, ch_l = ch["toffoli_count"], ch["num_qubits"], ch["lambda_eff"]
rot1 = (0.56 * math.log2(1 / iv["rotation_eps"]) + 5.3) / 4
dq_q = 1 - ((dqi_Q - 1) / dqi_Q) ** 2
xr = [  # (task, resource, relative cost change, same change in Toffolis of that task)
    ("invsqrt", "1 qubit", 1 / inv_Q, inv_C / inv_Q), ("invsqrt", "1 rotation", rot1 / inv_C, rot1), ("invsqrt", "1 Toffoli", 1 / inv_C, 1.0),
    ("dqi", "1 qubit (Q squared)", dq_q, dq_q * dqi_T), ("dqi", "1 failed shot fewer in a draw", 1 - R0 / rep_factor(cp_upper(1)), (1 - R0 / rep_factor(cp_upper(1))) * dqi_T),
    ("dqi", "100 Toffolis", 100 / dqi_T, 100.0), ("dqi", "1 Toffoli", 1 / dqi_T, 1.0),
    ("dfthcsos", "1 qubit", 1 / ch_Q, ch_T / ch_Q), ("dfthcsos", "0.01 Ha of lambda_eff", 0.01 / ch_l, 0.01 / ch_l * ch_T),
    ("dfthcsos", "1 loader Toffoli per pass", 2 / ch_T, 2.0), ("dfthcsos", "1 Toffoli", 1 / ch_T, 1.0),
]
pts_ = [gm * ((1 / (1 - rel)) ** (1 / 3) - 1) for _, _, rel, _ in xr]
fig, ax = plt.subplots(figsize=(12, 5.2))
for i, ((t_, res_, rel, tof), p_) in enumerate(zip(xr, pts_)):
    ax.plot([1e-6, p_], [i, i], color=RULE, lw=1.2, zorder=1); ax.scatter([p_], [i], s=70, color=TC[t_], edgecolor=SURF, linewidth=1.5, zorder=3)
    ax.text(p_ * 1.25, i, f"+{p_:.5f}   (= {tof:,.1f} Toffolis)" if tof != 1 else f"+{p_:.6f}", va="center", fontsize=10, color=INK)
ax.set_yticks(range(len(xr))); ax.set_yticklabels([f"{TN[t_]}: {r_}" for t_, r_, _, _ in xr]); ax.invert_yaxis()
ax.set_xscale("log"); ax.set_xlim(5e-6, 1); ax.set_xlabel("score points gained by removing one unit (log scale)"); ax.grid(axis="y", visible=False)
headline(fig, f"One qubit is worth ~{inv_C / inv_Q:.0f} Toffolis in invsqrt, ~{dq_q * dqi_T:,.0f} in DQI, ~{ch_T / ch_Q:.0f} in chemistry",
         "score points at the portfolio's operating point; the label converts each unit into Toffolis of the same task")
plt.show()
print(f"{'task':9s} {'resource':26s} {'cost change':>12s} {'score points':>13s} {'= Toffolis':>11s}")
for (t_, res_, rel, tof), p_ in zip(xr, pts_): print(f"{t_:9s} {res_:26s} {-100 * rel:11.5f}% {p_:+13.6f} {tof:11.1f}")
print(f"Reading: 0.001 points on this board is about {300 * 0.001 / gm:.4f}% of one task's cost "
      f"({0.003 / gm * dqi_T:,.0f} DQI Toffolis, {0.003 / gm * inv_C:.2f} invsqrt Toffoli-eq., {0.003 / gm * ch_T:.1f} chemistry Toffolis).")

<a id="s4"></a>
<div style="font-family: 'Segoe UI', Roboto, 'Helvetica Neue', Arial, sans-serif; background: #0b1f4b; color: #ffffff; padding: 10px 16px; border-radius: 8px; font-size: 22px; font-weight: 700;">3 &middot; What we tried and why it failed</div>

<div style="font-family: 'Segoe UI', Roboto, 'Helvetica Neue', Arial, sans-serif; font-size: 15.5px; line-height: 1.6; color: #111111; background: #ffffff;">
Between 11 and 14 November we attacked the best public file of every task. Most attempts failed, and the reasons are
more useful than the attempts, because they tell you where <i>not</i> to dig. Each card gives the idea, what we
measured, and why it stopped. The numbers come from our run logs (exact simulation on all inputs where that is
possible, otherwise the official verifier); estimates are marked as such. The exchange rates of section 2 convert
everything into score points: one invsqrt qubit is worth about 38 of its Toffolis, one DQI qubit about 860.
<br><span style="display: inline-block; margin-top: 6px;">
<span style="background: #0f7a4f; color: #ffffff; border-radius: 8px; padding: 2px 9px; font-size: 13px; font-weight: 700;">WORKED</span>
<span style="background: #b3261e; color: #ffffff; border-radius: 8px; padding: 2px 9px; font-size: 13px; font-weight: 700;">FAILED</span>
<span style="background: #7a4f00; color: #ffffff; border-radius: 8px; padding: 2px 9px; font-size: 13px; font-weight: 700;">NOT BUILT</span>
</span></div>

#### invsqrt (46 qubits; one qubit = 2.2% of the cost)

<div style="font-family: 'Segoe UI', Roboto, 'Helvetica Neue', Arial, sans-serif; font-size: 15px; line-height: 1.55; color: #111111; background: #ffffff; border: 1px solid #d9d9d9; border-left: 6px solid #0f7a4f; border-radius: 8px; padding: 9px 14px; margin: 8px 0;">
<div style="font-size: 16.5px; font-weight: 700;">A Toffoli whose result is never read <span style="background: #0f7a4f; color: #ffffff; border-radius: 8px; padding: 1px 8px; font-size: 12.5px;">WORKED &middot; in the submission</span></div>
In <code style="background: #eef1f6; color: #111111; padding: 1px 5px; border-radius: 4px; font-size: 92%;">dcde7078</code>, op 3292 <code style="background: #eef1f6; color: #111111; padding: 1px 5px; border-radius: 4px; font-size: 92%;">CCX q16 q20 q21</code> writes into <code style="background: #eef1f6; color: #111111; padding: 1px 5px; border-radius: 4px; font-size: 92%;">q21</code>, which is next touched only by <code style="background: #eef1f6; color: #111111; padding: 1px 5px; border-radius: 4px; font-size: 92%;">HMR q21 b61</code> at op 3403. Measuring first
and applying <code style="background: #eef1f6; color: #111111; padding: 1px 5px; border-radius: 4px; font-size: 92%;">CZ q16 q20 if b61</code> gives the same values, record and phase. All 3,133,452 inputs in exact simulation: restored, clean, same
phase margin (116,320,642 units), mean Toffolis 1,480.744 &rarr; 1,479.744. <b>Official verifier: 78,765.51 &rarr; 78,719.53.</b> Our earlier peephole
scan had reported zero such cases, because it skipped the input-register wires, and <code style="background: #eef1f6; color: #111111; padding: 1px 5px; border-radius: 4px; font-size: 92%;">q21</code> is one of them.
</div>

<div style="font-family: 'Segoe UI', Roboto, 'Helvetica Neue', Arial, sans-serif; font-size: 15px; line-height: 1.55; color: #111111; background: #ffffff; border: 1px solid #d9d9d9; border-left: 6px solid #b3261e; border-radius: 8px; padding: 9px 14px; margin: 8px 0;">
<div style="font-size: 16.5px; font-weight: 700;">45 qubits, three attempts <span style="background: #b3261e; color: #ffffff; border-radius: 8px; padding: 1px 8px; font-size: 12.5px;">FAILED &middot; adder carries</span></div>
Break-even is about 38 Toffolis for one qubit. On <code style="background: #eef1f6; color: #111111; padding: 1px 5px; border-radius: 4px; font-size: 92%;">321aff83</code> the circuit is at 46 wires for 8,137 operations in 485 episodes. An exact
rescheduler (swap elimination, renamed classical bits, 437 measurements moved earlier) shrank this to 1,792 units in 261 episodes, but 257 of them
are opened by the carry of an alternating-carry adder. In-place adders would cost +257 Toffolis; on <code style="background: #eef1f6; color: #111111; padding: 1px 5px; border-radius: 4px; font-size: 92%;">4560ac0c</code> the cheapest exact cover
(rebuilds plus single-carry adders) costs <b>+196</b>. Hiding the rare shift flag (set for 63 inputs) in affine parities of idle wires: zero candidates.
</div>

<div style="font-family: 'Segoe UI', Roboto, 'Helvetica Neue', Arial, sans-serif; font-size: 15px; line-height: 1.55; color: #111111; background: #ffffff; border: 1px solid #d9d9d9; border-left: 6px solid #b3261e; border-radius: 8px; padding: 9px 14px; margin: 8px 0;">
<div style="font-size: 16.5px; font-weight: 700;">Fewer or cheaper phase rotations <span style="background: #b3261e; color: #ffffff; border-radius: 8px; padding: 1px 8px; font-size: 12.5px;">FAILED &middot; the angles are already optimal</span></div>
The phase is exactly linear in the rotation angles, so a linear program finds the best angles. It gains 2 units of a 58.6M-unit margin on
<code style="background: #eef1f6; color: #111111; padding: 1px 5px; border-radius: 4px; font-size: 92%;">9c708f7b</code> and 116,291,417 &rarr; 116,320,642 on <code style="background: #eef1f6; color: #111111; padding: 1px 5px; border-radius: 4px; font-size: 92%;">4560ac0c</code> (cost &minus;0.08). Dropping one rotation and re-tuning the others: 32 of 33 are
infeasible; the one feasible drop costs 79,369.55 against 78,765.26. The smallest rotation weight (2<sup>&minus;36</sup> turn = 266M units) exceeds twice the margin; no angle can snap to T; no two rotations fire on the same inputs.
</div>

<div style="font-family: 'Segoe UI', Roboto, 'Helvetica Neue', Arial, sans-serif; font-size: 15px; line-height: 1.55; color: #111111; background: #ffffff; border: 1px solid #d9d9d9; border-left: 6px solid #b3261e; border-radius: 8px; padding: 9px 14px; margin: 8px 0;">
<div style="font-size: 16.5px; font-weight: 700;">Skip the recomputations of the backward half <span style="background: #b3261e; color: #ffffff; border-radius: 8px; padding: 1px 8px; font-size: 12.5px;">FAILED &middot; no free wire</span></div>
The uncompute half executes about 791 Toffolis against 689 forward (+102): 64 replays of a 13-wire divider register and about 69 conditional carry
fix-ups. Each is the price of a width saving. Every one of the 490 places where a product could be kept crosses at least 100 moments where no wire
is free (median 846); exact packing keeps 0 products at 46 qubits and 2 at 47, far below the ~38 a qubit costs.
</div>

<div style="font-family: 'Segoe UI', Roboto, 'Helvetica Neue', Arial, sans-serif; font-size: 15px; line-height: 1.55; color: #111111; background: #ffffff; border: 1px solid #d9d9d9; border-left: 6px solid #0f7a4f; border-radius: 8px; padding: 9px 14px; margin: 8px 0;">
<div style="font-size: 16.5px; font-weight: 700;">Splicing public circuits at clean cuts <span style="background: #0f7a4f; color: #ffffff; border-radius: 8px; padding: 1px 8px; font-size: 12.5px;">WORKED &middot; 12 Nov, since superseded</span></div>
Two circuits can be joined wherever both hold the same wires and phase and no classical bit crosses the cut. A shortest path over all such
points of the public 62-qubit circuits gave <code style="background: #eef1f6; color: #111111; padding: 1px 5px; border-radius: 4px; font-size: 92%;">9c708f7b</code>: 1,213.87 Toffolis per shot against 1,216.83 (official, &minus;0.20% cost). Two days later the
46-qubit line made the 62-qubit family obsolete. <b>Trap:</b> on 512 sample shots a splice looked 0.228 Toffoli-equivalents cheaper, on 4,098 only 0.109:
conditional blocks fire on random outcomes, so re-measure every candidate on a large fresh sample.
</div>

#### DQI (889 qubits; one qubit = 0.22% of the cost, about 860 Toffolis)

<div style="font-family: 'Segoe UI', Roboto, 'Helvetica Neue', Arial, sans-serif; font-size: 15px; line-height: 1.55; color: #111111; background: #ffffff; border: 1px solid #d9d9d9; border-left: 6px solid #b3261e; border-radius: 8px; padding: 9px 14px; margin: 8px 0;">
<div style="font-size: 16.5px; font-weight: 700;">Cheaper field multipliers <span style="background: #b3261e; color: #ffffff; border-radius: 8px; padding: 1px 8px; font-size: 12.5px;">FAILED &middot; already at the best known rank</span></div>
We rebuilt the decoder with a logger on every Toffoli (384,003, exact). 81% are bilinear field products: GF(2<sup>12</sup>) at rank 42 (161,322 Toffolis),
GF(2<sup>24</sup>) at rank 99 (136,224), and smaller fields at their exact optimum. 42 is the best known bound for GF(2<sup>12</sup>) (Ballet et al. survey).
The only gap is one rank-126 product that could be 99: <b>27 Toffolis, 0.007%</b>. Squarings and linear maps are already free CX.
</div>

<div style="font-family: 'Segoe UI', Roboto, 'Helvetica Neue', Arial, sans-serif; font-size: 15px; line-height: 1.55; color: #111111; background: #ffffff; border: 1px solid #d9d9d9; border-left: 6px solid #b3261e; border-radius: 8px; padding: 9px 14px; margin: 8px 0;">
<div style="font-size: 16.5px; font-weight: 700;">Keep products instead of recomputing them on the way back <span style="background: #b3261e; color: #ffffff; border-radius: 8px; padding: 1px 8px; font-size: 12.5px;">FAILED &middot; 0 at 889 qubits</span></div>
The raw-query returns recompute 620 GF(2<sup>24</sup>) products (61,380 Toffolis). They belong to a three-term recurrence that runs in place (a bijection),
so there is no garbage to measure away. Keeping one needs 24 qubits across the whole middle of the decoder, where no wire is free. Keeping the 82
repeated 12-bit norms (3,444 Toffolis) needs +12 qubits, about &minus;10k Toffolis in value.
</div>

<div style="font-family: 'Segoe UI', Roboto, 'Helvetica Neue', Arial, sans-serif; font-size: 15px; line-height: 1.55; color: #111111; background: #ffffff; border: 1px solid #d9d9d9; border-left: 6px solid #b3261e; border-radius: 8px; padding: 9px 14px; margin: 8px 0;">
<div style="font-size: 16.5px; font-weight: 700;">Skip the rare routing gates that look like no-ops <span style="background: #b3261e; color: #ffffff; border-radius: 8px; padding: 1px 8px; font-size: 12.5px;">FAILED &middot; they are live</span></div>
On 1,024 typical shots, 9,724 of 12,240 controlled swaps never swap anything. We simulated 65,536 shots from the official distribution: 948 of the 1,020
routing slots of the key phases <b>do</b> fire, on about 1 in 4,096 frames (an abnormal Euclid step), so dropping one would fail about 2.5e-4 of shots.
Skipping them only when rare would need a classical bit that depends on the data, and kickmix has none: <code style="background: #eef1f6; color: #111111; padding: 1px 5px; border-radius: 4px; font-size: 92%;">HMR</code> outcomes are uniformly random.
</div>

<div style="font-family: 'Segoe UI', Roboto, 'Helvetica Neue', Arial, sans-serif; font-size: 15px; line-height: 1.55; color: #111111; background: #ffffff; border: 1px solid #d9d9d9; border-left: 6px solid #7a4f00; border-radius: 8px; padding: 9px 14px; margin: 8px 0;">
<div style="font-size: 16.5px; font-weight: 700;">One qubit less (892 &rarr; 891, then 889 &rarr; 888) <span style="background: #7a4f00; color: #ffffff; border-radius: 8px; padding: 1px 8px; font-size: 12.5px;">NOT BUILT &middot; plans only</span></div>
On <code style="background: #eef1f6; color: #111111; padding: 1px 5px; border-radius: 4px; font-size: 92%;">eae396ba</code>, 635 places reach 892 live wires; splitting a retained product (+1 Toffoli each) covered 48 of them. On <code style="background: #eef1f6; color: #111111; padding: 1px 5px; border-radius: 4px; font-size: 92%;">821bf8d1</code>, 509 peak
episodes remain after free rebuilds, in seven families with a one-wire reallocation each (estimate: +0.010 to +0.021 points). Writing it needs the
decoder's source build, not a gate-level patch of the 53M-operation file, so no exact 888-qubit file exists.
</div>

<div style="font-family: 'Segoe UI', Roboto, 'Helvetica Neue', Arial, sans-serif; font-size: 15px; line-height: 1.55; color: #111111; background: #ffffff; border: 1px solid #d9d9d9; border-left: 6px solid #b3261e; border-radius: 8px; padding: 9px 14px; margin: 8px 0;">
<div style="font-size: 16.5px; font-weight: 700;">Raise the extractor cutoff further <span style="background: #b3261e; color: #ffffff; border-radius: 8px; padding: 1px 8px; font-size: 12.5px;">FAILED beyond 18</span></div>
Each cutoff step removes 116 static Toffolis and admits more rare error patterns. Official runs at 17 and 18: PASS with 0 failures. By the
verifier's weight law, 19 adds 3.46e-6 failures per shot, 0.35 per board draw, and one failure costs about 0.041 points: negative. The submission's
DQI is already at 18 (Ert&#287;rul &Ouml;zer's idea).
</div>

<div style="font-family: 'Segoe UI', Roboto, 'Helvetica Neue', Arial, sans-serif; font-size: 15px; line-height: 1.55; color: #111111; background: #ffffff; border: 1px solid #d9d9d9; border-left: 6px solid #b3261e; border-radius: 8px; padding: 9px 14px; margin: 8px 0;">
<div style="font-size: 16.5px; font-weight: 700;">Repair the rare decoder failures <span style="background: #b3261e; color: #ffffff; border-radius: 8px; padding: 1px 8px; font-size: 12.5px;">FAILED &middot; no cheap exact fix</span></div>
On <code style="background: #eef1f6; color: #111111; padding: 1px 5px; border-radius: 4px; font-size: 92%;">7cbe8ecf</code> (the same decoder at cutoff 15), 25 salts &times; 102,400 official shots gave 6 failures (2.3e-6 per shot), all predicted by the builder's
own classifier; replaying 175 flagged shots: 128 fail, deterministically (any measurement seed). One failure in the board draw costs 0.041 points,
the expected loss is about 0.019 (estimate). A file with one changed byte gets a fresh draw, which is luck, not a fix.
</div>

<div style="font-family: 'Segoe UI', Roboto, 'Helvetica Neue', Arial, sans-serif; font-size: 15px; line-height: 1.55; color: #111111; background: #ffffff; border: 1px solid #d9d9d9; border-left: 6px solid #0f7a4f; border-radius: 8px; padding: 9px 14px; margin: 8px 0;">
<div style="font-size: 16.5px; font-weight: 700;">Readout windows and product retention <span style="background: #0f7a4f; color: #ffffff; border-radius: 8px; padding: 1px 8px; font-size: 12.5px;">WORKED &middot; small, earlier files</span></div>
Moving Atakan Aldemir's two readout-window regions onto <code style="background: #eef1f6; color: #111111; padding: 1px 5px; border-radius: 4px; font-size: 92%;">bb4287e7</code> saved 48.28 Toffolis per shot (official; figure below). Renaming scratch
lifetimes to retain more products found 2 extra on <code style="background: #eef1f6; color: #111111; padding: 1px 5px; border-radius: 4px; font-size: 92%;">4c81b726</code>: official PASS at 0 failures on three salts, &minus;1.58 Toffolis per shot. Real, verified, and tiny.
</div>

#### Chemistry (the loaders are 55% of the Toffolis; the factorization sets the rest)

<div style="font-family: 'Segoe UI', Roboto, 'Helvetica Neue', Arial, sans-serif; font-size: 15px; line-height: 1.55; color: #111111; background: #ffffff; border: 1px solid #d9d9d9; border-left: 6px solid #b3261e; border-radius: 8px; padding: 9px 14px; margin: 8px 0;">
<div style="font-size: 16.5px; font-weight: 700;">Re-plan the loaders <span style="background: #b3261e; color: #ffffff; border-radius: 8px; padding: 1px 8px; font-size: 12.5px;">FAILED &middot; at the floor of the method</span></div>
A loader needs about 42 products per block of three low monomials by parameter counting, 210 for a generic loader: the public ones use 212 (193 for
loader 4). With full mixing the counting floor is about 25 lower (estimate), but no construction gets there: a block matcher reached 229, greedy
240-250, and removing any one product or pair leaves the target uncovered, so local and flip-graph search have no move.
</div>

<div style="font-family: 'Segoe UI', Roboto, 'Helvetica Neue', Arial, sans-serif; font-size: 15px; line-height: 1.55; color: #111111; background: #ffffff; border: 1px solid #d9d9d9; border-left: 6px solid #b3261e; border-radius: 8px; padding: 9px 14px; margin: 8px 0;">
<div style="font-size: 16.5px; font-weight: 700;">31 "dead" loader Toffolis <span style="background: #b3261e; color: #ffffff; border-radius: 8px; padding: 1px 8px; font-size: 12.5px;">FAILED &middot; rejected by the verifier</span></div>
Simulated with the angle registers at zero, 31 loader Toffolis never mattered. The official verifier rejected the edit ("uncorrected phase"):
loader k starts from the words of batch k&minus;1, and those Toffolis read word bits that are nonzero there. <b>The input model is part of every claim.</b>
</div>

<div style="font-family: 'Segoe UI', Roboto, 'Helvetica Neue', Arial, sans-serif; font-size: 15px; line-height: 1.55; color: #111111; background: #ffffff; border: 1px solid #d9d9d9; border-left: 6px solid #b3261e; border-radius: 8px; padding: 9px 14px; margin: 8px 0;">
<div style="font-size: 16.5px; font-weight: 700;">A smaller factorization shape at fixed angle words <span style="background: #b3261e; color: #ffffff; border-radius: 8px; padding: 1px 8px; font-size: 12.5px;">FAILED &middot; error floor above the gate</span></div>
One column less (C = 44 &rarr; 43) would cut the cost by 0.53% with unchanged loaders. Minimising the two-body error alone, the public table's own words
reach 19.797 mHa at C = 44 (the table sits only 0.20 mHa above that floor); at C = 43 the floor is 21.10 mHa, above the 20 mHa gate. One group less
(R = 8 &rarr; 7) reaches 256 mHa. A full rebuild at C = 40 might give about +0.16 points (estimate); we did not try it.
</div>

In [ ]:
# Figure: four walls, drawn from the numbers in the cards above (our run logs).
fig, axs = plt.subplots(2, 2, figsize=(13.5, 9.2))
ax = axs[0, 0]  # invsqrt 45 qubits
lab = ["in-place adders\n(321aff83)", "cheapest exact cover\n(4560ac0c)", "rebuilds for 127 of\n261 episodes only"]
val = [257, 196, 61]
ax.barh(lab, val, color=[C[7], C[7], "#e8a09d"], height=0.6); ax.invert_yaxis()
ax.axvline(37.8, color=INK, lw=1.6, ls="--"); ax.text(41, -0.42, "break-even 37.8", fontsize=10.5, color=INK, va="bottom")
for i, x in enumerate(val): ax.text(x + 4, i, f"+{x}", va="center", fontsize=11, color=INK, fontweight="bold")
ax.set_ylim(2.5, -0.75); ax.set_xlim(0, 300); ax.set_xlabel("extra Toffolis per shot to free one qubit"); ax.grid(axis="y", visible=False)
ax.set_title("invsqrt, 45 qubits: the carries cost 5x the prize", loc="left")
ax = axs[0, 1]  # rotations
lab = ["drop rot. 18\n(the smallest)", "drop rot. 19", "drop rot. 23", "drop rot. 20"]
val = [-48.5, -120, -295, -474]
ax.barh(lab, val, color=C[7], height=0.6); ax.invert_yaxis()
ax.axvline(23.47, color=INK, lw=1.6, ls="--"); ax.text(30, -0.45, "needed:\n+23.5M", fontsize=10.5, color=INK, va="bottom")
ax.axvline(0, color=INK, lw=1)
for i, x in enumerate(val): ax.text(x - 8, i, f"{x:g}M", va="center", ha="right", fontsize=11, color=INK, fontweight="bold")
ax.set_ylim(3.5, -0.8); ax.set_xlim(-640, 140); ax.set_xlabel("best margin after the drop (millions of 2^-64 turn)"); ax.grid(axis="y", visible=False)
ax.set_title("invsqrt rotations: every drop goes negative", loc="left")
ax = axs[1, 0]  # DQI attribution
parts_ = [("GF(2^12) products, rank 42", 161322, True), ("GF(2^24) products, rank 99", 136224, True), ("smaller fields (exact optimum)", 10050 + 2052 + 1062, True),
          ("one rank-126 product (could be 99)", 126, False), ("root scan, routing, phases, tests", 73167, False)]
y = np.arange(len(parts_))
ax.barh(y, [p[1] for p in parts_], color=[C[1] if p[2] else "#f0b49b" for p in parts_], height=0.62); ax.invert_yaxis()
ax.set_yticks(y); ax.set_yticklabels([p[0] for p in parts_])
for i, p in enumerate(parts_): ax.text(p[1] + 2500, i, f"{p[1]:,}" + ("  (at best known rank)" if p[2] else ""), va="center", fontsize=10.5, color=INK)
ax.set_xlim(0, 270000); ax.xaxis.set_major_formatter(FuncFormatter(lambda x, _: f"{x / 1000:.0f}k")); ax.grid(axis="y", visible=False)
ax.set_xlabel("static Toffolis of the decoder (384,003, attributed by a logged rebuild)")
ax.set_title("DQI: 81% of the Toffolis are already optimal products", loc="left")
ax = axs[1, 1]  # chemistry error floor
lab = ["public table\nC = 44", "floor of its\nwords, C = 44", "floor of its\nwords, C = 43"]
val = [19.999998, 19.797, 21.099]
ax.bar(lab, val, color=[C[2], C[2], C[7]], width=0.55)
ax.axhline(20, color=INK, lw=1.6, ls="--"); ax.text(1.0, 20.25, "gate 20 mHa", fontsize=10.5, color=INK, ha="right", va="bottom")
for i, x in enumerate(val): ax.text(i, x + 0.04, f"{x:.6f}" if i == 0 else f"{x:.3f}", ha="center", va="bottom", fontsize=11, color=INK, fontweight="bold")
ax.set_ylim(19, 21.6); ax.set_ylabel("two-body error (mHa)"); ax.grid(axis="x", visible=False)
ax.set_title("chemistry: one column less fails the gate", loc="left")
ax.text(0.02, 0.97, "R = 7 at fixed words: 256 mHa (off scale)", transform=ax.transAxes, fontsize=10.5, color=INK, va="top")
headline(fig, "Four walls: why the obvious levers are closed",
         "numbers from our runs (exact simulation, linear programs, a logged DQI rebuild, an error-minimising refit); see the cards above")
plt.show()

### What worked: the dead write, drawn and checked
The rule behind our one change is the measurement-based uncomputation trick of section 1, read backwards: if a
Toffoli's target is only measured afterwards, the Toffoli's only lasting effect is a sign (&minus;1)<sup>m&middot;ab</sup>,
and a free classically controlled `CZ` applies exactly that sign. The cell below checks both circuits with `kmsim` on
a large sample that includes the tightest input; the full proof on all 3,133,452 inputs takes `qccscope` six seconds
(section 4).

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(12.5, 3.3))
Wd = ["q16", "q20", "q21", "b:b61"]
n0 = draw_circuit(axs[0], Wd, [(0, "CCX", "q16", "q20", "q21"), (1.3, "BOX", "q16", "q21", {"label": "q21\nidle"}), (2.6, "HMR", "q21", "b:b61")], 3.6)
n1 = draw_circuit(axs[1], Wd, [(0, "HMR", "q21", "b:b61"), (1, "CZ", "q16", "q20", {"if": "b:b61"}), (2.3, "BOX", "q16", "q21", {"label": "q21\nidle"})], 3.6)
axs[0].set_title(f"before (dcde7078, op 3292): {n0} Toffoli", loc="left"); axs[1].set_title(f"after (our fix): {n1} Toffolis", loc="left")
headline(fig, "Our fix: measure first, then a free CZ replaces the Toffoli", "box = ops 3293-3402, where q21 is not touched; same values, record and phase on every input and every measurement outcome")
plt.show()
rng_d = np.random.default_rng(14)
xs_d = np.unique(np.concatenate([np.arange(1, 20001), rng_d.integers(1, 3133453, 60000), [159, 161, 228663, 3133452]])).astype(np.int64)
for name, path in (("dcde7078 (public)", INV_ORIG), ("with our fix", INV)):
    r_, _ = check(path, xs_d, seed=1)
    print(f"{name:18s}: {r_['nq']} qubits, restored {r_['restored']}, dirty {r_['dirty']}, worst error/tolerance {r_['worst_ratio']:.6f}, "
          f"mean Toffolis {r_['toff_mean']:,.3f} ({len(xs_d):,} inputs)")
print(f"official verifier on the fixed file in this run: cost {v['invsqrt']['score']:,.2f}, mean Toffolis {v['invsqrt']['toffoli_count']:,.4f} "
      f"(public dcde7078: 78,765.51, 1,480.7497)")

### What worked earlier: the readout-window transplant (DQI, 11 November)
A *readout window* is `CSWAP c a b` followed by measuring both outputs. The same phase and classical record come from
`CX a b; HMR a -> m1; CZ c b if m1; HMR b -> m2; BIT_INVERT m1 if m2`, with no Toffoli. Atakan Aldemir's `e9f54be3`
line had two regions with such windows; we moved them onto his `bb4287e7`. Official verifier, default salt (our run of
12 November; these files are history, the submission's DQI is `821bf8d1`):

<table style="font-family: 'Segoe UI', Roboto, 'Helvetica Neue', Arial, sans-serif; font-size: 14.5px; color: #111111; border-collapse: collapse; background: #ffffff;">
<tr style="background: #e3ecf8;"><th style="background: #ffffff; color: #111111; text-align: left; padding: 5px 9px;">build on <code style="background: #eef1f6; color: #111111; padding: 1px 5px; border-radius: 4px; font-size: 92%;">bb4287e7</code></th><th style="background: #ffffff; color: #111111; text-align: right; padding: 5px 9px;">Toffolis per shot</th><th style="background: #ffffff; color: #111111; text-align: right; padding: 5px 9px;">vs base</th><th style="background: #ffffff; color: #111111; text-align: right; padding: 5px 9px;">value errors</th><th style="background: #ffffff; color: #111111; text-align: right; padding: 5px 9px;">phase errors</th></tr>
<tr style="background: #ffffff; color: #111111;"><td style="background: #ffffff; color: #111111; padding: 5px 9px;">base</td><td style="background: #ffffff; color: #111111; text-align: right; padding: 5px 9px;">394,165.16</td><td style="background: #ffffff; color: #111111; text-align: right; padding: 5px 9px;">0.00</td><td style="background: #ffffff; color: #111111; text-align: right; padding: 5px 9px;">0</td><td style="background: #ffffff; color: #111111; text-align: right; padding: 5px 9px;">0</td></tr>
<tr style="background: #ffffff; color: #111111;"><td style="background: #ffffff; color: #111111; padding: 5px 9px;">+ window region A</td><td style="background: #ffffff; color: #111111; text-align: right; padding: 5px 9px;">394,141.10</td><td style="background: #ffffff; color: #111111; text-align: right; padding: 5px 9px;">&minus;24.06</td><td style="background: #ffffff; color: #111111; text-align: right; padding: 5px 9px;">1</td><td style="background: #ffffff; color: #111111; text-align: right; padding: 5px 9px;">0</td></tr>
<tr style="background: #ffffff; color: #111111;"><td style="background: #ffffff; color: #111111; padding: 5px 9px;">+ window region B</td><td style="background: #ffffff; color: #111111; text-align: right; padding: 5px 9px;">394,140.99</td><td style="background: #ffffff; color: #111111; text-align: right; padding: 5px 9px;">&minus;24.17</td><td style="background: #ffffff; color: #111111; text-align: right; padding: 5px 9px;">1</td><td style="background: #ffffff; color: #111111; text-align: right; padding: 5px 9px;">0</td></tr>
<tr style="background: #ffffff; color: #111111;"><td style="background: #ffffff; color: #111111; padding: 5px 9px;"><b>+ both (our transplant)</b></td><td style="background: #ffffff; color: #111111; text-align: right; padding: 5px 9px;"><b>394,116.88</b></td><td style="background: #ffffff; color: #111111; text-align: right; padding: 5px 9px;"><b>&minus;48.28</b></td><td style="background: #ffffff; color: #111111; text-align: right; padding: 5px 9px;">1</td><td style="background: #ffffff; color: #111111; text-align: right; padding: 5px 9px;">0</td></tr>
<tr style="background: #ffffff; color: #111111;"><td style="background: #ffffff; color: #111111; padding: 5px 9px;">+ both + a retained-product region</td><td style="background: #ffffff; color: #111111; text-align: right; padding: 5px 9px;">394,116.02</td><td style="background: #ffffff; color: #111111; text-align: right; padding: 5px 9px;">&minus;49.14</td><td style="background: #ffffff; color: #111111; text-align: right; padding: 5px 9px;">1</td><td style="background: #ffffff; color: #111111; text-align: right; padding: 5px 9px;"><b style="color: #b3261e;">1,754</b></td></tr>
</table>

Each window saves exactly 24 Toffolis on every shot. The retained-product region alone breaks the phase on 1.7% of
shots, because it depends on two companion regions that this line had rewritten: **transplants must move whole
dependency groups, and must be verified.** (The single value error is the old line's rare failure, section 5.2.)

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(12.5, 3.4), gridspec_kw={"width_ratios": [4.6, 7.2]})
n_b = draw_circuit(axs[0], ["c", "a", "b", "b:m1", "b:m2"], [(0, "CSWAP", "c", "a", "b"), (1, "HMR", "a", "b:m1"), (2, "HMR", "b", "b:m2")], 3)
n_a = draw_circuit(axs[1], ["c", "a", "b", "b:m1", "b:m2"], [(0, "CX", "a", "b"), (1, "HMR", "a", "b:m1"), (2, "CZ", "c", "b", {"if": "b:m1"}),
                                                            (3, "HMR", "b", "b:m2"), (4, "NOT", "b:m1", {"if": "b:m2"})], 5)
axs[0].set_title(f"before: {n_b} Toffoli", loc="left"); axs[1].set_title(f"after: {n_a} Toffolis (NOT = BIT_INVERT m1 if m2)", loc="left")
headline(fig, "A readout window without its Toffoli: swap, then measure = measure, then fix the sign",
         "both leave c unchanged, a and b at 0, and the phase (-1)^(m1 a' + m2 b') of the swapped values")
plt.show()
before = "CSWAP q0 q1 q2\nHMR q1 b0\nHMR q2 b1"
after = "CX q1 q2\nHMR q1 b0\nCZ q0 q2 if b0\nHMR q2 b1\nBIT_INVERT b0 if b1"
ok = True
for c_ in (0, 1):
    for a in (0, 1):
        for b in (0, 1):
            ap, bp = (b, a) if c_ else (a, b)
            for txt_ in (before, after):
                s_ = run_text(txt_, [c_, a, b], 3, 2, batch=256, seed=4 * c_ + 2 * a + b)
                expect = ((s_.b[0] & bool(ap)) ^ (s_.b[1] & bool(bp))).astype(np.uint64) << np.uint64(63)
                ok &= bool((s_.ph == expect).all()) and not s_.q[1:].any() and bool((s_.q[0] == bool(c_)).all())
print("readout-window identity holds on all 8 inputs x 256 random measurement records:", ok)

<a id="s5"></a>
<div style="font-family: 'Segoe UI', Roboto, 'Helvetica Neue', Arial, sans-serif; background: #0b1f4b; color: #ffffff; padding: 10px 16px; border-radius: 8px; font-size: 22px; font-weight: 700;">4 &middot; How to find improvements</div>

<div style="font-family: 'Segoe UI', Roboto, 'Helvetica Neue', Arial, sans-serif; font-size: 15.5px; line-height: 1.6; color: #111111; background: #ffffff;">
Once the cheap levers are gone, guessing is expensive. Performance engineers in the same situation <b>profile first, bound
second, and search last</b>. This is the procedure we now follow, and the one that found the dead write:
</div>

<table style="font-family: 'Segoe UI', Roboto, 'Helvetica Neue', Arial, sans-serif; font-size: 14.5px; color: #111111; border-collapse: collapse; background: #ffffff;">
<tr style="background: #e3ecf8;"><th style="background: #ffffff; color: #111111; text-align: left; padding: 5px 9px;">Step</th><th style="background: #ffffff; color: #111111; text-align: left; padding: 5px 9px;">Question</th><th style="background: #ffffff; color: #111111; text-align: left; padding: 5px 9px;">Borrowed from</th></tr>
<tr style="background: #ffffff; color: #111111;"><td style="background: #ffffff; color: #111111; padding: 5px 9px;"><b>1. Exchange rates</b></td><td style="background: #ffffff; color: #111111; padding: 5px 9px;">What is one qubit, one Toffoli, one rotation, one bit of phase margin worth in points? (section 2)</td><td style="background: #ffffff; color: #111111; padding: 5px 9px;">sensitivity analysis</td></tr>
<tr style="background: #f6f8fb;"><td style="background: #ffffff; color: #111111; padding: 5px 9px;"><b>2. Census and attribution</b></td><td style="background: #ffffff; color: #111111; padding: 5px 9px;">Which gadget pays for each Toffoli? Descend only into the dominant one.</td><td style="background: #ffffff; color: #111111; padding: 5px 9px;">top-down profiling (Yasin 2014), USE method</td></tr>
<tr style="background: #ffffff; color: #111111;"><td style="background: #ffffff; color: #111111; padding: 5px 9px;"><b>3. Width profile</b></td><td style="background: #ffffff; color: #111111; padding: 5px 9px;">Where is the qubit peak, which wires open each peak episode, which wires sit idle there?</td><td style="background: #ffffff; color: #111111; padding: 5px 9px;">pebble games; conditionally clean ancillae (Khattar &amp; Gidney 2024)</td></tr>
<tr style="background: #f6f8fb;"><td style="background: #ffffff; color: #111111; padding: 5px 9px;"><b>4. Distance to a bound</b></td><td style="background: #ffffff; color: #111111; padding: 5px 9px;">How far is each gadget from its best known cost (bilinear rank, n&minus;1 for an adder, LP-optimal angles, a counting floor)? A gadget at its bound is closed.</td><td style="background: #ffffff; color: #111111; padding: 5px 9px;">roofline model (Williams et al. 2009)</td></tr>
<tr style="background: #ffffff; color: #111111;"><td style="background: #ffffff; color: #111111; padding: 5px 9px;"><b>5. Amdahl ceiling</b></td><td style="background: #ffffff; color: #111111; padding: 5px 9px;">If the lever worked perfectly, what is the most it could give? (the pale bars of the first figure)</td><td style="background: #ffffff; color: #111111; padding: 5px 9px;">Amdahl 1967</td></tr>
<tr style="background: #f6f8fb;"><td style="background: #ffffff; color: #111111; padding: 5px 9px;"><b>6. Motifs and redundancy</b></td><td style="background: #ffffff; color: #111111; padding: 5px 9px;">Match known patterns; fingerprint values on many inputs to find repeated, constant, affine and dead Toffolis.</td><td style="background: #ffffff; color: #111111; padding: 5px 9px;">superoptimisation, e-graphs (as a cheap proxy)</td></tr>
<tr style="background: #ffffff; color: #111111;"><td style="background: #ffffff; color: #111111; padding: 5px 9px;"><b>7. Check against history</b></td><td style="background: #ffffff; color: #111111; padding: 5px 9px;">Is this a negative result already measured? Do not redo it.</td><td style="background: #ffffff; color: #111111; padding: 5px 9px;">lab notebooks</td></tr>
<tr style="background: #f6f8fb;"><td style="background: #ffffff; color: #111111; padding: 5px 9px;"><b>8. Fail fast, then verify</b></td><td style="background: #ffffff; color: #111111; padding: 5px 9px;">Run the cheapest falsification first; count nothing until the official verifier passes it.</td><td style="background: #ffffff; color: #111111; padding: 5px 9px;">the 31 "dead" loader Toffolis (section 3)</td></tr>
</table>

### The pattern catalog in brief
Only three things cost anything: Toffoli-like gates (<code style="background: #eef1f6; color: #111111; padding: 1px 5px; border-radius: 4px; font-size: 92%;">CCX</code>, <code style="background: #eef1f6; color: #111111; padding: 1px 5px; border-radius: 4px; font-size: 92%;">CCZ</code>, <code style="background: #eef1f6; color: #111111; padding: 1px 5px; border-radius: 4px; font-size: 92%;">CSWAP</code>, counted where they run), non-Clifford
rotations (invsqrt only) and the qubit peak. The catalog lists, for each motif, the best known construction and the
gate-stream signature a detector can look for:

<table style="font-family: 'Segoe UI', Roboto, 'Helvetica Neue', Arial, sans-serif; font-size: 14.5px; color: #111111; border-collapse: collapse; background: #ffffff;">
<tr style="background: #e3ecf8;"><th style="background: #ffffff; color: #111111; text-align: left; padding: 5px 9px;">Motif</th><th style="background: #ffffff; color: #111111; text-align: left; padding: 5px 9px;">Signature in a .kmx file</th><th style="background: #ffffff; color: #111111; text-align: left; padding: 5px 9px;">Best known</th><th style="background: #ffffff; color: #111111; text-align: left; padding: 5px 9px;">In the current files</th></tr>
<tr style="background: #ffffff; color: #111111;"><td style="background: #ffffff; color: #111111; padding: 5px 9px;">temporary AND</td><td style="background: #ffffff; color: #111111; padding: 5px 9px;"><code style="background: #eef1f6; color: #111111; padding: 1px 5px; border-radius: 4px; font-size: 92%;">CCX a b t</code> ... <code style="background: #eef1f6; color: #111111; padding: 1px 5px; border-radius: 4px; font-size: 92%;">CCX a b t</code></td><td style="background: #ffffff; color: #111111; padding: 5px 9px;">1 Toffoli; erase by <code style="background: #eef1f6; color: #111111; padding: 1px 5px; border-radius: 4px; font-size: 92%;">HMR</code> + <code style="background: #eef1f6; color: #111111; padding: 1px 5px; border-radius: 4px; font-size: 92%;">CZ if m</code> (Gidney 2018)</td><td style="background: #ffffff; color: #111111; padding: 5px 9px;">0 left anywhere</td></tr>
<tr style="background: #f6f8fb;"><td style="background: #ffffff; color: #111111; padding: 5px 9px;"><b>dead write</b></td><td style="background: #ffffff; color: #111111; padding: 5px 9px;"><code style="background: #eef1f6; color: #111111; padding: 1px 5px; border-radius: 4px; font-size: 92%;">CCX a b t</code>, then only writes to t, then <code style="background: #eef1f6; color: #111111; padding: 1px 5px; border-radius: 4px; font-size: 92%;">HMR t</code></td><td style="background: #ffffff; color: #111111; padding: 5px 9px;">0 Toffolis: <code style="background: #eef1f6; color: #111111; padding: 1px 5px; border-radius: 4px; font-size: 92%;">HMR</code> first, <code style="background: #eef1f6; color: #111111; padding: 1px 5px; border-radius: 4px; font-size: 92%;">CZ a b if m</code></td><td style="background: #ffffff; color: #111111; padding: 5px 9px;"><b>1 in invsqrt: our fix</b></td></tr>
<tr style="background: #ffffff; color: #111111;"><td style="background: #ffffff; color: #111111; padding: 5px 9px;">constant or equal controls</td><td style="background: #ffffff; color: #111111; padding: 5px 9px;">a control that is 0 (or equal to the other) on every input</td><td style="background: #ffffff; color: #111111; padding: 5px 9px;">delete, or a free <code style="background: #eef1f6; color: #111111; padding: 1px 5px; border-radius: 4px; font-size: 92%;">CX</code></td><td style="background: #ffffff; color: #111111; padding: 5px 9px;">0 on all 3,133,452 invsqrt inputs</td></tr>
<tr style="background: #f6f8fb;"><td style="background: #ffffff; color: #111111; padding: 5px 9px;">repeated product</td><td style="background: #ffffff; color: #111111; padding: 5px 9px;">two Toffolis with equal operand values on every input</td><td style="background: #ffffff; color: #111111; padding: 5px 9px;">keep it on a spare wire (retention)</td><td style="background: #ffffff; color: #111111; padding: 5px 9px;">479 in invsqrt, 0 fit at 46 qubits</td></tr>
<tr style="background: #ffffff; color: #111111;"><td style="background: #ffffff; color: #111111; padding: 5px 9px;">n-bit adder</td><td style="background: #ffffff; color: #111111; padding: 5px 9px;">ladder of <code style="background: #eef1f6; color: #111111; padding: 1px 5px; border-radius: 4px; font-size: 92%;">CCX</code> into fresh carry wires</td><td style="background: #ffffff; color: #111111; padding: 5px 9px;">n&minus;1 Toffolis; in place +1 per step (Cuccaro 2004)</td><td style="background: #ffffff; color: #111111; padding: 5px 9px;">sets the invsqrt peak</td></tr>
<tr style="background: #f6f8fb;"><td style="background: #ffffff; color: #111111; padding: 5px 9px;">GF(2<sup>k</sup>) product</td><td style="background: #ffffff; color: #111111; padding: 5px 9px;">one layer of r ANDs of XOR combinations</td><td style="background: #ffffff; color: #111111; padding: 5px 9px;">bilinear rank: 42 for k = 12 (best known)</td><td style="background: #ffffff; color: #111111; padding: 5px 9px;">at the bound in DQI</td></tr>
<tr style="background: #ffffff; color: #111111;"><td style="background: #ffffff; color: #111111; padding: 5px 9px;">QROM loader</td><td style="background: #ffffff; color: #111111; padding: 5px 9px;">cache of address monomials, then products into word bits</td><td style="background: #ffffff; color: #111111; padding: 5px 9px;">counting floor of the decomposition</td><td style="background: #ffffff; color: #111111; padding: 5px 9px;">at the floor of its method</td></tr>
<tr style="background: #f6f8fb;"><td style="background: #ffffff; color: #111111; padding: 5px 9px;">rotations</td><td style="background: #ffffff; color: #111111; padding: 5px 9px;"><code style="background: #eef1f6; color: #111111; padding: 1px 5px; border-radius: 4px; font-size: 92%;">Z_POW q t</code> with t not a multiple of 1/4</td><td style="background: #ffffff; color: #111111; padding: 5px 9px;">LP-optimal angles; Hamming-weight phasing for equal angles</td><td style="background: #ffffff; color: #111111; padding: 5px 9px;">LP-optimal; 2 equal pairs only</td></tr>
</table>

### A demo of <code style="background: #eef1f6; color: #111111; padding: 1px 5px; border-radius: 4px; font-size: 92%;">qccscope</code>, our analysis library
<code style="background: #eef1f6; color: #111111; padding: 1px 5px; border-radius: 4px; font-size: 92%;">qccscope</code> (about 1,650 lines of pure Python and numpy, no kickmix) parses a <code style="background: #eef1f6; color: #111111; padding: 1px 5px; border-radius: 4px; font-size: 92%;">.kmx</code> file, takes a census, computes the
width profile, runs the motif detectors and a bit-sliced value scan, and prices every finding in leaderboard points.
For invsqrt the value scan is exhaustive: all 3,133,452 inputs in a few seconds. The next cell installs it from an
embedded archive (Apache-2.0, like the rest of this notebook's code), then runs it on the **public** circuit
<code style="background: #eef1f6; color: #111111; padding: 1px 5px; border-radius: 4px; font-size: 92%;">dcde7078</code>, before our fix.

In [ ]:
# qccscope: install from the embedded archive (source of the package, zipped and base64-encoded).
import base64, io
QCCSCOPE_B64 = "..."
QS_DIR = pathlib.Path("/tmp/qccscope_pkg")
with zipfile.ZipFile(io.BytesIO(base64.b64decode(QCCSCOPE_B64))) as z: z.extractall(QS_DIR)
sys.path.insert(0, str(QS_DIR))
from qccscope import report as QR
print("qccscope installed:", sorted(p.name for p in (QS_DIR / "qccscope").glob("*.py")))

In [ ]:
t_ = time.time()
r0 = QR.analyse(str(INV_ORIG), "invsqrt", exhaustive=True)
r1 = QR.analyse(str(INV), "invsqrt", exhaustive=True)
print(f"qccscope on dcde7078: {r0['circ'].n_ops:,} operations, {r0['circ'].num_qubits} qubits, value scan over {r0['V'].lanes:,} inputs, "
      f"end state {r0['V'].end_ok}; {time.time() - t_:.1f} s for both files")
F = sorted(r0["findings"], key=lambda f: -f["points"])
show = [f for f in F if f["count"] or f["confidence"] != "exact"][:9]
print(f"\n{'id':5s} {'finding':62s} {'count':>6s} {'Toffolis':>9s} {'qubits':>6s} {'points':>8s}  confidence")
for f in show:
    print(f"{f['id']:5s} {f['motif'][:62]:62s} {f['count']:6d} {f['d_toffoli']:9.1f} {f['d_qubits']:6d} {f['points']:+8.4f}  {f['confidence']}")
dw = r0["S"].dead_writes
print(f"\ndead writes found: {dw} (Toffoli op, measuring op); after our fix: {r1['S'].dead_writes}")
print(f"executed Toffolis per shot, exact over all inputs: {r0['V'].toffoli_exec_mean:,.4f} -> {r1['V'].toffoli_exec_mean:,.4f}")

In [ ]:
lv = r0["lv"]; circ = r0["circ"]
from qccscope import kmx as QK
tof = np.flatnonzero(np.isin(circ.op, QK.TOFFOLI_OPS)); zp = np.flatnonzero(circ.op == QK.ZPOW)
fig, axs = plt.subplots(2, 1, figsize=(12.5, 5.6), sharex=True, gridspec_kw={"height_ratios": [2.2, 1]})
ax = axs[0]
ax.plot(lv.width, color=C[0], lw=1.2); ax.axhline(lv.peak, color=INK, lw=1, ls="--")
ax.text(len(lv.width) * 0.995, lv.peak + 0.3, f"peak {lv.peak}: {len(lv.episodes())} episodes", ha="right", va="bottom", fontsize=10.5, color=INK)
ax.set_ylim(20, lv.peak + 3); ax.set_ylabel("live wires")
for k0, k1 in dw:
    ax.axvline(k0, color=C[7], lw=2); ax.annotate(f"dead write: CCX at op {k0},\ntarget only measured at op {k1}", (k0, 30), (k0 + 1800, 25), fontsize=10.5, color=INK,
                                                   arrowprops=dict(arrowstyle="-|>", color=INK), bbox=dict(fc="#fff4f2", ec=C[7], boxstyle="round,pad=0.3"))
ax.set_title("width profile (static liveness)", loc="left")
ax = axs[1]
h, e = np.histogram(tof, bins=120, range=(0, circ.n_ops)); ax.bar(e[:-1], h, width=e[1] - e[0], align="edge", color=C[1], label="Toffoli-like gates")
for k in zp: ax.axvline(k, color=C[6], lw=0.6, alpha=0.7)
ax.set_ylabel("Toffolis\nper bin"); ax.set_xlabel("operation index"); ax.set_title(f"where the {len(tof):,} Toffoli-like gates sit; purple lines = {len(zp)} Z_POW", loc="left")
headline(fig, "qccscope on the public invsqrt: one exact finding, at op 3292",
         f"dcde7078, all {r0['V'].lanes:,} inputs simulated; the 46-wire peak is opened by adder carries in {len(lv.episodes())} short episodes")
plt.show()

<a id="s6"></a>
<div style="font-family: 'Segoe UI', Roboto, 'Helvetica Neue', Arial, sans-serif; background: #0b1f4b; color: #ffffff; padding: 10px 16px; border-radius: 8px; font-size: 22px; font-weight: 700;">5 &middot; Deep dives</div>

### 5.0 <code style="background: #eef1f6; color: #111111; padding: 1px 5px; border-radius: 4px; font-size: 92%;">kmsim</code>: an exact numpy simulator for these circuits
<code style="background: #eef1f6; color: #111111; padding: 1px 5px; border-radius: 4px; font-size: 92%;">kmsim</code> (a hidden cell near the top, ~150 lines, Apache-2.0) is what we use to test circuit surgery in seconds before
an official run: <code style="background: #eef1f6; color: #111111; padding: 1px 5px; border-radius: 4px; font-size: 92%;">parse(path)</code> reads a <code style="background: #eef1f6; color: #111111; padding: 1px 5px; border-radius: 4px; font-size: 92%;">.kmx</code> file, <code style="background: #eef1f6; color: #111111; padding: 1px 5px; border-radius: 4px; font-size: 92%;">Sim</code> runs a batch of shots as numpy boolean arrays (one row per
wire, one column per shot) and can stop after any operation through a <code style="background: #eef1f6; color: #111111; padding: 1px 5px; border-radius: 4px; font-size: 92%;">hook</code>, and <code style="background: #eef1f6; color: #111111; padding: 1px 5px; border-radius: 4px; font-size: 92%;">check(path, xs)</code> applies the
invsqrt verifier's rules: input restored, scratch clean, no bad <code style="background: #eef1f6; color: #111111; padding: 1px 5px; border-radius: 4px; font-size: 92%;">R</code>, phase within tolerance, mean Toffolis per shot.

<div style="font-family: 'Segoe UI', Roboto, 'Helvetica Neue', Arial, sans-serif; font-size: 15px; color: #111111; background: #eaf7f0; border-left: 5px solid #0f7a4f; padding: 10px 14px; border-radius: 6px; margin: 6px 0;">
<b>Validation.</b> On the submission's invsqrt circuit, kmsim gives the same qubit count, the same worst error over
tolerance and (up to measurement randomness) the same mean Toffolis as the official verifier, which runs 200.5M shots.
</div>

In [ ]:
rng = np.random.default_rng(0)
xs = np.concatenate([np.arange(1, 4001), rng.integers(1, 3133453, 12000), np.array([159, 161, 221, 644, 228663, 3099851, 3133452])]).astype(np.int64)
t = time.time(); r_inv, s_inv = check(INV, xs, seed=1); dt = time.time() - t
print("kmsim   :", {k: r_inv[k] for k in ("nq", "restored", "dirty", "worst_ratio", "toff_mean")}, f"({dt:.1f} s for {len(xs):,} shots)")
print("verifier:", {k: v["invsqrt"].get(k) for k in ("num_qubits", "worst_error_over_tolerance", "toffoli_count")})
if BASE is not None:
    r_base, _ = check(BASE, xs[:4000], seed=1)
    print("hosts' reference circuit:", {k: r_base[k] for k in ("nq", "worst_ratio", "toff_mean")})

### 5.1 invsqrt

**The recipe** (the hosts' paper, Sec. III.A): shift x into a fixed range, get a first guess of 1/&radic;x from a
lookup table plus a cubic polynomial, refine it with one Newton&ndash;Raphson step (the fixed-point cousin of the
*Quake III* fast inverse square root), shift back, and kick the result into the phase with `Z_POW` rotations, one
per bit, before cleaning up. The reference does this with 260 qubits. The public line went from 260 to 62 qubits
and then, in November, to 46 (Syed Asad Ali's own circuit and Ali Haydar &Ouml;zda&#287;'s `4560ac0c`); Oleg Sher's
*Quantum Circuits from Zero* walks through the design step by step. Below, `kmsim` opens the submission's circuit.

In [ ]:
# Figure: the invsqrt pipeline (the hosts' reference recipe), with the reference's size.
fig, ax = plt.subplots(figsize=(12, 2.5)); ax.set_xlim(0, 10.5); ax.set_ylim(0, 2); ax.axis("off")
steps = [("|x>", "22-qubit input"), ("normalise", "shift x by 2s\ninto [1, 4)"), ("first guess", "lookup table +\ncubic polynomial"),
         ("Newton step", "y1 = y0 (3 - x y0^2) / 2"), ("shift back", "y = y1 * 2^s"), ("phase kickback", "Z_POW per bit:\ne^(2 pi i y)"), ("uncompute", "scratch -> |0>\n(HMR + fix-ups)")]
for i, (a, b) in enumerate(steps):
    x0 = 0.1 + i * 1.48; col = TC["invsqrt"] if 0 < i < 6 else NAVY
    ax.add_patch(FancyBboxPatch((x0, 0.55), 1.22, 0.95, boxstyle="round,pad=0.03", facecolor=col, edgecolor=col, alpha=0.95 if i in (0, 6) else 0.85))
    ax.text(x0 + 0.61, 1.22, a, ha="center", va="center", color="white", fontsize=11, fontweight="bold")
    ax.text(x0 + 0.61, 0.83, b, ha="center", va="center", color="white", fontsize=9)
    if i < len(steps) - 1: ax.annotate("", (x0 + 1.46, 1.02), (x0 + 1.24, 1.02), arrowprops=dict(arrowstyle="-|>", color=INK2))
ax.text(0.1, 0.18, "hosts' reference: 260 qubits, 7,735 Toffolis, 37 rotations, C = 7,988.   Portfolio: "
        f"{v['invsqrt']['num_qubits']} qubits, {v['invsqrt']['toffoli_count']:,.1f} Toffolis, {v['invsqrt']['rotation_count']:.0f} rotations, C = {v['invsqrt']['toffoli_equivalent']:,.1f}.", fontsize=10, color=INK2)
headline(fig, "invsqrt: a fixed-point fast inverse square root, run reversibly", "the reference recipe (paper Sec. III.A); the public circuits keep this shape")
plt.show()

#### One shot, every wire
A kickmix shot is just bits moving on wires. The raster shows all 46 wires (rows) along the circuit (columns) for one
input, the tightest one for the error budget: dark = the wire holds a 1. The 22 input wires stay put at the top; the
scratch wires fill, compute and empty again.

In [ ]:
# Figure: the state of every wire along one shot of the submission's invsqrt circuit.
ops, regs = parse(INV)
nb_ = 1 + max(max((max(o[2]) for o in ops if o[2]), default=0), max((o[3] for o in ops if o[3] is not None), default=0))
nq_ = 1 + max(max(o[1]) for o in ops if o[1])
x_one = v["invsqrt"]["worst_x"]; s1 = Sim(ops, nq_, nb_, 1, 0)
for i, qi in enumerate(regs["r0"]): s1.q[qi] = (x_one >> i) & 1 == 1
step = 8; snaps = []
s1.run(hook=lambda k, sim: snaps.append(sim.q[:, 0].copy()) if k % step == 0 else None)
order = list(regs["r0"]) + [q for q in range(nq_) if q not in regs["r0"]]
img = np.array(snaps).T[order]
fig, ax = plt.subplots(figsize=(12, 4.0))
from matplotlib.colors import ListedColormap
ax.imshow(img, aspect="auto", interpolation="nearest", cmap=ListedColormap([SURF, TC["invsqrt"]]), extent=(0, len(ops), nq_ - 0.5, -0.5))
ax.axhline(len(regs["r0"]) - 0.5, color=C[1], lw=1)
ax.set_xlabel("operation index"); ax.set_ylabel("wire"); ax.grid(False)
headline(fig, f"One shot of the invsqrt circuit (x = {x_one}): the scratch wires fill and empty again",
         f"{nq_} wires x {len(ops):,} operations (every {step}th shown); blue = the wire holds 1; above the orange line: the 22 input wires")
plt.show()
print("x restored and scratch clean at the end:", int(sum(int(s1.q[q, 0]) << i for i, q in enumerate(regs["r0"]))) == x_one and not s1.q[order[22:], 0].any())

#### Where the Toffoli-equivalents go
The verifier's cost is Q x C with C = mean Toffolis + T/4, every non-Clifford rotation priced at 0.56 log2(1/eps) + 5.3
T, where eps splits the error margin left over by the simulated error evenly across the rotations. Below: cumulative
Toffolis along the circuit, with the rotations marked, and the qubits in use (non-zero on at least one shot).

In [ ]:
xs_a = np.concatenate([np.arange(1, 1501), rng.integers(1, 3133453, 2500)]).astype(np.int64)
s = Sim(ops, nq_, nb_, len(xs_a), 3)
for i, qi in enumerate(regs["r0"]): s.q[qi] = (xs_a >> i) & 1 == 1
cum_t = np.zeros(len(ops)); live = np.zeros(len(ops), dtype=int)
def hook(k, sim):
    cum_t[k] = sim.toff.mean()
    if k % 5 == 0: live[k] = int(sim.q.any(axis=1).sum())
s.run(hook=hook)
cum_t = np.maximum.accumulate(cum_t)  # condition push/pop operations skip the hook
live = np.where(live > 0, live, np.nan).astype(float)
rot_idx = [k for k, o in enumerate(ops) if o[0] == "Z_POW" and o[4] % (1 << 61) != 0]
fig, axs = plt.subplots(2, 1, figsize=(12, 5.4), sharex=True)
axs[0].plot(cum_t, color=TC["invsqrt"]); axs[0].set_ylabel("Toffolis per shot")
for k in rot_idx: axs[0].axvline(k, color=C[3], lw=0.7, alpha=0.8)
axs[0].text(rot_idx[0], cum_t[-1] * 0.86, f" {len(rot_idx)} rotations (yellow lines)", color=INK2, ha="right", fontsize=10.5)
axs[0].set_title("cumulative mean Toffolis", loc="left", fontsize=11.5)
ok = ~np.isnan(live); axs[1].plot(np.flatnonzero(ok), live[ok], color=C[2], lw=1.2); axs[1].set_ylim(20, nq_ + 2)
axs[1].set_ylabel("wires"); axs[1].set_xlabel("operation index"); axs[1].set_title(f"wires holding data on at least one shot (out of {nq_})", loc="left", fontsize=11.5)
headline(fig, "invsqrt: the Toffolis are spread along the circuit, the rotations sit in one stretch",
         "submission circuit, 4,000 shots in kmsim")
plt.show()
n_rot = len(rot_idx); rot_cost = n_rot * (0.56 * math.log2(1 / v["invsqrt"]["rotation_eps"]) + 5.3) / 4
print(f"{n_rot} rotations at eps={v['invsqrt']['rotation_eps']:.2e}: {rot_cost:.0f} Toffoli-eq "
      f"= {100 * rot_cost / v['invsqrt']['toffoli_equivalent']:.1f}% of C; one qubit = {100 / nq_:.2f}% of the cost")

#### Error budget across x
The worst input uses almost all of its tolerance, so the rotations get almost no synthesis budget (their eps is tiny,
which is why each costs ~29 T). Every error source has been squeezed to just under tol(x) = 2^-24 / sqrt(x) turns.

In [ ]:
xs_e = np.unique(np.concatenate([np.arange(1, 3000), np.geomspace(1, 3133452, 6000).astype(np.int64), [v["invsqrt"]["worst_x"]]])).astype(np.int64)
_, se = check(INV, xs_e, seed=2)
target, tol = target_and_tol(xs_e)
ratio = np.abs((se.ph - target).view(np.int64)).astype(float) / tol.astype(float)
fig, ax = plt.subplots(figsize=(12, 3.4))
ax.scatter(xs_e, ratio, s=4, color=TC["invsqrt"], alpha=0.5, linewidths=0); ax.set_xscale("log"); ax.axhline(1, color=C[7], lw=1)
ax.text(1.2, 1.02, "fail", color=C[7], fontsize=10.5, va="bottom")
iw = int(np.argmax(ratio)); ax.scatter([xs_e[iw]], [ratio[iw]], s=50, color=INK, zorder=4); ax.annotate(f"x = {xs_e[iw]}: {ratio[iw]:.4f}", (xs_e[iw], ratio[iw]), (xs_e[iw] * 0.08, 0.86), fontsize=10.5, color=INK, bbox=dict(fc=SURF, ec=RULE, boxstyle="round,pad=0.25"), arrowprops=dict(arrowstyle="-", color=INK2))
ax.set_xlabel("x"); ax.set_ylabel("|phase error| / tolerance"); ax.set_ylim(0, 1.12)
headline(fig, f"invsqrt error budget: the worst input sits at {100 * ratio.max():.1f}% of its tolerance", "simulated phase error before rotation synthesis, 9,000 inputs")
plt.show()

#### The invsqrt landscape: every public circuit in qubits x Toffoli-equivalents
Each point is a public circuit from manwithacat's dataset (scan of 2026-11-10), measured with `kmsim` on the same
4,000 inputs; the grey curves are constant cost Q x C. For weeks the frontier was a dense cluster at 62 qubits that
fought over C; the 46-qubit line (not in that scan; the orange dot is the submission) jumped off the cluster by cutting
16 qubits while paying more Toffolis per shot, which the exchange rate of section 2 says is the right trade.

In [ ]:
land = []
xs_l = np.concatenate([np.arange(1, 1001), np.random.default_rng(4).integers(1, 3133453, 3000), [159, 161, 3133452]]).astype(np.int64)
rot1 = (0.56 * math.log2(1 / v["invsqrt"]["rotation_eps"]) + 5.3) / 4
for name, path in list(PUB_KMX.items()) + [("submission", INV)]:
    try: r_, s_ = check(path, xs_l, seed=1)
    except Exception: continue
    if not (r_["restored"] and r_["dirty"] == 0 and r_["worst_ratio"] <= 1): continue
    ops_l, _ = parse(path); n_rot_ = sum(1 for o in ops_l if o[0] == "Z_POW" and o[4] % (1 << 61))
    land.append((name, r_["nq"], r_["toff_mean"] + n_rot_ * rot1))
L = np.array([(q, c) for _, q, c in land], dtype=float); is_pf = np.array([n == "submission" for n, _, _ in land])
fig, axs = plt.subplots(1, 2, figsize=(12.5, 5.6), gridspec_kw={"width_ratios": [1.15, 1]})
ax = axs[0]
qq = np.linspace(40, 290, 300); y_lo, y_hi = L[:, 1].min() * 0.85, L[:, 1].max() * 1.5
for sc in (1e5, 2e5, 4e5, 1e6, 2e6):
    ax.plot(qq, sc / qq, color="#cfcfcf", lw=1)
    if y_lo < sc / 280 < y_hi: ax.text(280, sc / 280, f"Q x C = {sc:,.0f}", color=INK2, fontsize=9.5, ha="right", va="bottom")
ax.scatter(L[~is_pf, 0], L[~is_pf, 1], s=30, color=TC["invsqrt"], edgecolor=SURF, linewidth=1.2, zorder=3, label="public circuit (dataset)")
ax.scatter(L[is_pf, 0], L[is_pf, 1], s=110, color=C[1], zorder=4, label="submission (46-qubit line + our fix)")
ax.set_xscale("log"); ax.set_yscale("log"); ax.set_xlabel("qubits Q"); ax.set_ylabel("Toffoli-equivalents C")
ax.xaxis.set_major_locator(FixedLocator([46, 62, 100, 150, 260])); ax.xaxis.set_major_formatter(FormatStrFormatter("%g")); ax.xaxis.set_minor_locator(FixedLocator([]))
ax.yaxis.set_major_locator(FixedLocator([1500, 2000, 3000, 5000, 8000])); ax.yaxis.set_major_formatter(FormatStrFormatter("%g")); ax.yaxis.set_minor_locator(FixedLocator([]))
ax.set_xlim(40, 290); ax.set_ylim(y_lo, y_hi); ax.legend(loc="upper left"); ax.set_title("all valid public circuits", loc="left")
ax = axs[1]
QC = L[:, 0] * L[:, 1]; order_ = np.argsort(QC)[:15]
ax.barh(range(len(order_)), QC[order_], color=[C[1] if is_pf[i] else TC["invsqrt"] for i in order_], height=0.7); ax.invert_yaxis()
ax.set_yticks(range(len(order_))); ax.set_yticklabels([("submission" if is_pf[i] else land[i][0][:8]) + f" ({int(L[i, 0])} q)" for i in order_], fontsize=10)
ax.set_xlim(QC[order_].min() * 0.9, QC[order_].max() * 1.02); ax.xaxis.set_major_formatter(FuncFormatter(lambda x, _: f"{x / 1000:.0f}k"))
ax.set_xlabel("cost Q x C (kmsim estimate)"); ax.grid(axis="y", visible=False); ax.set_title("the 15 cheapest", loc="left")
headline(fig, f"invsqrt landscape: the 46-qubit line beat the 62-qubit cluster by {100 * (1 - QC[is_pf][0] / QC[~is_pf].min()):.0f}%",
         f"{(~is_pf).sum()} valid public circuits from manwithacat's dataset, measured with kmsim on 4,000 inputs")
plt.show()

### 5.2 DQI
**The decoder** (paper Sec. III.B): from the 70 syndromes of a Reed&ndash;Solomon code over GF(2<sup>12</sup>), run the
extended Euclidean algorithm reversibly to get the error locator, sweep the 4,095 positions to recover each error
value and apply its sign, then run everything backwards to restore the syndromes. The reference uses 1,893 qubits and
5.7M Toffolis. The submission's file `821bf8d1` uses 889 qubits and about 382,600 Toffolis per shot: Syed Asad Ali's
from-source decoder at Ert&#287;rul &Ouml;zer's extractor cutoff 18, with Areeb Ali's 241 retained products. Because
the cost has **Q squared**, one DQI qubit is worth about 860 of its Toffolis. Section 3 shows where its Toffolis go.

#### Rare failures and board luck
The decoder is approximate on a tiny set of error patterns (abnormal Euclid steps the builder does not handle at the
extractor cutoff). The verifier draws 102,400 shots per file from the file's hash and a secret salt, and every failed
shot raises the repetition factor R. Our measurements on the same decoder at cutoff 15 (`7cbe8ecf`): the builder's
own classifier predicts 5.3e-6 failing shots per shot; 25 official salts gave 6 failures in 2,560,000 shots
(2.3e-6); replaying 175 flagged shots, 128 failed (estimated true rate 3.3e-6). Cutoff 18 adds about 7.7e-7 by the
verifier's weight law. The figure prices a board draw of the submission with a rate of 3.3e-6 + 7.7e-7 (an estimate).

In [ ]:
eps = 3.3e-6 + 7.7e-7; lam = eps * 102400
if "km" in globals():  # cross-check our R formula against the verifier module (it imports kickmix)
    sys.path.insert(0, str(VER / "dqi")); import verify_submission_dqi as vq
    assert all(abs(vq.repetition_overhead(vq.clopper_pearson_upper_bound(k, 102400, -128.0)) - rep_factor(cp_upper(k))) < 1e-9 for k in range(5))
probs, scores = [], []
for k in range(5):
    pk = math.exp(-lam) * lam ** k / math.factorial(k)
    dq_ = rows["dqi"][2] / R0 * rep_factor(cp_upper(k))
    sc = (REF["invsqrt"] / rows["invsqrt"][2] * REF["dqi"] / dq_ * REF["dfthcsos"] / rows["dfthcsos"][2]) ** (1 / 3)
    probs.append(pk); scores.append(sc)
fig, ax = plt.subplots(figsize=(12, 3.6))
ax.bar([f"{k} failed" for k in range(5)], probs, color=TC["dqi"], width=0.55)
for k, (p_, s_) in enumerate(zip(probs, scores)): ax.text(k, p_, f"score {s_:.4f}", ha="center", va="bottom", color=INK, fontsize=11)
ax.set_ylabel("probability"); ax.set_ylim(0, max(probs) * 1.2); ax.grid(axis="x", visible=False)
headline(fig, f"A board draw of the submission's DQI: P(zero failures) = {probs[0]:.2f} (estimate)", f"Poisson with eps = {eps:.1e} per shot; label = this run's score if its DQI drew that many failures")
plt.show()
print(f"this run (default salt): value errors {v['dqi']['value_errors']}, phase errors {v['dqi']['phase_errors']}, R {v['dqi']['repetition_factor']:.4f}; "
      f"expected failures per draw {lam:.2f}; expected score {sum(p * s for p, s in zip(probs, scores)) / sum(probs):.5f}")

#### The DQI landscape: public files in qubits x Toffolis
Our [QCC Circuit Atlas](https://www.kaggle.com/datasets/asteriospanoras/qcc-circuit-atlas) indexes the public files too (scan of
2026-11-12); the points below come from manwithacat's dataset (scan of 2026-11-10), the black dot is the submission's
file `821bf8d1` (889 qubits, newer than that scan). Cost = R x Q<sup>2</sup> x T, so the grey lines of equal cost are T proportional to 1/Q<sup>2</sup>. Families sit on
vertical lines (one qubit count each); moving one qubit left is worth ~860 Toffolis.

In [ ]:
dq_pts = []
for r in man:
    if r["task"] != "dqi": continue
    c_ = fnum(r["dqi_zero_failure_cost"]) if r.get("dqi_zero_failure_cost") else fnum(r["local_task_score"]); q_ = fnum(r["qubits"])
    if c_ == c_ and q_ == q_ and c_ < 1e12: dq_pts.append((q_, c_ / (R0 * q_ * q_)))
D = np.array(dq_pts); D = D[(D[:, 0] >= 885) & (D[:, 0] <= 900)]
fig, ax = plt.subplots(figsize=(12, 4.3))
x0, x1 = min(D[:, 0].min(), rows['dqi'][0]) - 1, D[:, 0].max() + 1; y0, y1 = min(D[:, 1].min(), rows["dqi"][1]) * 0.985, np.percentile(D[:, 1], 85)
qq = np.linspace(x0, x1, 100)
for sc in (3.75e11, 3.9e11, 4.1e11, 4.4e11):
    ax.plot(qq, sc / (R0 * qq ** 2), color=RULE, lw=1)
    yr = sc / (R0 * x1 ** 2)
    if y0 < yr < y1: ax.text(x1, yr, f" {sc:.2e}", color=INK2, fontsize=9.5, va="center")
    else:
        xa = math.sqrt(sc / (R0 * y1 * 0.97))
        if x0 < xa < x1: ax.text(xa, y1 * 0.97, f"{sc:.2e} ", color=INK2, fontsize=9.5, ha="right")
ax.set_xlim(x0, x1 + 2.5); ax.set_ylim(y0, y1)
ax.scatter(D[:, 0], D[:, 1], s=26, color=TC["dqi"], edgecolor=SURF, linewidth=1.2, zorder=3, label="public file")
ax.scatter([rows["dqi"][0]], [rows["dqi"][1]], s=90, color=INK, zorder=4, label="submission (821bf8d1)")
ax.yaxis.set_major_formatter(FuncFormatter(lambda y, _: f"{y / 1e3:,.0f}k"))
ax.set_xlabel("qubits Q"); ax.set_ylabel("Toffolis per shot T"); ax.legend(loc="lower right", fontsize=10)
headline(fig, "DQI landscape: families sit on fixed qubit counts; 889 qubits broke away", f"{len(D)} public files (manwithacat's dataset, costs at zero failed shots); grey = equal R Q^2 T")
plt.show()

### 5.3 Chemistry
**The block encoding** (paper Sec. III.C): a DF-THC factorization of the FeMoCo Hamiltonian (shape R, B, C) is loaded
row by row. Each row a of the factorization is a unit vector in 76 orbitals, written as 75 Givens angles of `brot`
bits on a fixed tree. The submitted loaders `ANGLE_k` stream these angles in batches of J words:
`|a>|z> -> |a>|z XOR delta_k(a)>` with `delta_k = batch_k XOR batch_(k-1)`, forward in U and backwards in U-dagger.
The state preparations and the Givens rotations themselves are not kickmix circuits; the verifier prices them with a
formula from (R, B, C) and `brot`. The score is lambda_eff x T x Q of the whole thing (section 2 shows the split).

#### The formula part: the shape (R, B, C) moves T x Q in steps
Only the ANGLE loaders are simulated. Below, the verifier's own cost model (`costs_prepare.py`) evaluated over the
basis size B at fixed R and C, and over R and C at fixed B, with the portfolio's loaders held fixed. Accuracy is not
modelled here: any other shape must still pass the 20 mHa gates with its own refit.

In [ ]:
sys.path.insert(0, str(VER / "dfthcsos" / "scoreinfra"))
from costs_prepare import SpinFreeData_V1 as Model
res = v["dfthcsos"]["resources"]
W_shape = np.load(NPZ)["W"].shape
print("factorization shape (R, B, C):", W_shape, " brot:", res["brot"], " J:", v["dfthcsos"].get("batch_size"), " loaders:", v["dfthcsos"].get("num_circuits"))
def model_cost(R, B, C, angles_T=res["angles_toffoli"], ang_pers=res["angles_persistent_qubits"], ang_tmp=res["angles_temporary_qubits"], beta=res["brot"]):
    m = Model(R=R, B=B, C=C, N=76, chi=10, beta=beta)
    o, i = m.outer_state_prep_qubit_cost(), m.inner_state_prep_qubit_cost()
    T = m.outer_state_prep() + m.inner_state_prep() + m.givens_rotations() + angles_T
    return T * (o["persistent"] + i["persistent"] + ang_pers + max(o["temporary"], i["temporary"], ang_tmp))
R0_, B0, C0 = W_shape; base_c = model_cost(R0_, B0, C0)
assert abs(base_c - v["dfthcsos"]["toffoli_count"] * v["dfthcsos"]["num_qubits"]) < 1e-6 * base_c
Bs = list(range(100, 200)); rel = []
for B in Bs:
    try: rel.append(model_cost(R0_, B, C0) / base_c)
    except ValueError: rel.append(np.nan)
rel = np.array(rel)
Rs, Cs = np.arange(4, 17), np.arange(30, 61, 2)
grid = np.full((len(Rs), len(Cs)), np.nan)
for i, R_ in enumerate(Rs):
    for j, C_ in enumerate(Cs):
        try: grid[i, j] = model_cost(int(R_), B0, int(C_)) / base_c
        except ValueError: pass
fig, axs = plt.subplots(1, 2, figsize=(12, 4.0), gridspec_kw={"width_ratios": [1.2, 1]})
ax = axs[0]; ax.step(Bs, rel, where="mid", color=TC["dfthcsos"]); ax.axvline(B0, color=C[1], lw=1.2); ax.axhline(1, color=RULE, lw=1)
ax.text(B0, np.nanmax(rel), f" portfolio B = {B0}", color=INK2, va="top", fontsize=10.5)
ax.set_xlabel(f"basis size B (R = {R0_}, C = {C0}, loaders fixed)"); ax.set_ylabel("T x Q relative to the portfolio")
ax.set_title("over B", loc="left", fontsize=11.5)
ax = axs[1]; im = ax.imshow(grid, origin="lower", aspect="auto", cmap="Greens", extent=(Cs[0] - 1, Cs[-1] + 1, Rs[0] - 0.5, Rs[-1] + 0.5))
ax.scatter([C0], [R0_], s=60, color=C[1], edgecolor=SURF, linewidth=1.5, zorder=3); ax.annotate("portfolio", (C0, R0_), (C0 + 3, R0_ + 2.5), fontsize=10, color=INK, arrowprops=dict(arrowstyle="-", color=INK2))
ax.set_xlabel("C (columns)"); ax.set_ylabel("R (ranks)"); ax.grid(False); cb = plt.colorbar(im, ax=ax, pad=0.02); cb.set_label("T x Q relative")
ax.set_title(f"over R and C (B = {B0})", loc="left", fontsize=11.5)
nxt = rel[Bs.index(B0) + 1] if B0 + 1 in Bs else np.nan
headline(fig, "Chemistry formula: T x Q moves in steps as the factorization shape changes",
         f"verifier's SpinFreeData_V1 model, loaders held fixed; B = {B0 + 1} would cost {100 * (nxt - 1):+.1f}%; accuracy (20 mHa gates) not modelled")
plt.show()
print(f"over B in [100, 199]: min {np.nanmin(rel):.3f}, max {np.nanmax(rel):.3f} of the portfolio's T x Q; "
      f"over R in [4, 16], C in [30, 60]: min {np.nanmin(grid):.3f}, max {np.nanmax(grid):.3f}")
print("""Reading: the formula part moves in steps (the lookup block sizes inside the formula change at thresholds such as B = 128),
and it moves with R and C too. Any cheaper shape must still pass the accuracy gates, which the portfolio already
presses against 20 mHa (section 1): finding one is a numerical factorization problem, not circuit surgery.""")

#### The loaders: cache monomials, two parities, one Toffoli per product
A loader must XOR, into 40 word bits (J = 5 words of `brot` = 8 bits in the submission), a fixed Boolean function of the
11 address bits. The public loaders (maoxin123's line, building on Lera's, syed3000's, gengsr's and others' work,
section 8) do it in two phases, which the miniature below draws and the bars count:
1. **Cache**: `CCX` builds ~75 products of address bits (monomials) on scratch wires, one Toffoli each.
2. **Products**: free `CX` parities build two linear forms (of address bits and cached monomials) on two helper wires;
   one `CCX` multiplies them into a word bit, and free `CX` fan-outs copy it to other word bits that need the same term.

At the end the cache is cleared with `HMR` and classically controlled `CZ` (zero Toffolis, section 1). Every Toffoli
saved per pass is worth 2 Toffolis of T (the loaders run forward and in reverse).

In [ ]:
# Figure: a loader in miniature (schematic) and the Toffoli split of the portfolio's 15 loaders (counted in the files).
fig, axs = plt.subplots(1, 2, figsize=(12.5, 4.4), gridspec_kw={"width_ratios": [1.25, 1]})
WL = ["a_i", "a_j", "cache", "f1", "f2", "w", "w'", "b:m"]
draw_circuit(axs[0], WL, [(0, "CCX", "a_i", "a_j", "cache"), (1, "CX", "a_i", "f1"), (2, "CX", "cache", "f1"), (3, "CX", "a_j", "f2"),
                          (4, "CCX", "f1", "f2", "w"), (5, "CX", "w", "w'"), (6, "BOX", "f1", "f2", {"label": "undo\nparities"}),
                          (7, "HMR", "cache", "b:m"), (8, "CZ", "a_i", "a_j", {"if": "b:m"})], 9)
axs[0].set_title("schematic: cache, parities, product, fan-out, free clean-up", loc="left", fontsize=11.5)
split = []
for p in CHEM_KMX:
    names_, regq = {}, collections.defaultdict(set)
    for line in open(p):
        t = line.split()
        if not t: continue
        if t[0] == "REGISTER": names_[t[1]] = t[2].strip('"')
        elif t[0] == "APPEND_TO_REGISTER": regq[t[2]].add(int(t[1][1:]))
    words = set().union(*[regq[r] for r, nm in names_.items() if nm.startswith("word")])
    o_, _ = parse(p); prod = sum(1 for o in o_ if o[0] == "CCX" and o[1][2] in words); allt = sum(1 for o in o_ if o[0] in TOFF)
    split.append((allt - prod, prod))
S = np.array(split); ks = np.arange(len(S)); ax = axs[1]
ax.bar(ks, S[:, 0], color=TC["dfthcsos"], width=0.75, label="cache (monomials)", edgecolor=SURF, linewidth=1.5)
ax.bar(ks, S[:, 1], bottom=S[:, 0], color=TC["dfthcsos"], alpha=0.45, width=0.75, label="products into word bits", edgecolor=SURF, linewidth=1.5)
for k in ks: ax.text(k, S[k].sum(), f"{S[k].sum()}", ha="center", va="bottom", fontsize=9, color=INK2)
ax.set_xticks(ks); ax.set_xlabel("loader k"); ax.set_ylabel("static Toffolis per pass")
ax.set_ylim(0, S.sum(axis=1).max() * 1.3); ax.legend(loc="upper right", ncol=2, fontsize=10); ax.grid(axis="x", visible=False)
ax.set_title(f"portfolio loaders: {S.sum():,} Toffolis per pass", loc="left", fontsize=11.5)
headline(fig, "Chemistry loaders: one Toffoli per product, after a shared cache of monomials",
         f"{S[:, 1].sum():,} product Toffolis and {S[:, 0].sum():,} cache Toffolis per pass; receipt: {res['angles_toffoli_per_pass']:,.0f} per pass, x2 passes")
plt.show()

#### The chemistry landscape: lambda_eff against T x Q
Each public chemistry submission is a point: the factorization sets lambda_eff (and, through the formula, part of T
and Q), the loaders set the rest. The gains came first from the factorization (new shapes and refits), then from the
loaders.

In [ ]:
ch_pts = [(fnum(r["lambda_eff"]), fnum(r["T_total"]) * fnum(r["Q_total"]), r["release_date"][:10]) for r in man if r["task"] == "chemistry"]
ch_pts = [p for p in ch_pts if p[0] == p[0] and p[1] == p[1]]
P = np.array([(a, b) for a, b, _ in ch_pts]); days = np.array([np.datetime64(d) for *_, d in ch_pts]).astype("datetime64[D]")
dnum = (days - days.min()).astype(float)
fig, ax = plt.subplots(figsize=(12, 4.4))
sc_ = ax.scatter(P[:, 1], P[:, 0], c=dnum, cmap="Greens", vmin=-dnum.max() * 0.3, s=26, edgecolor="#b9b8b3", linewidth=0.5, zorder=3)
ax.scatter([v["dfthcsos"]["toffoli_count"] * v["dfthcsos"]["num_qubits"]], [v["dfthcsos"]["lambda_eff"]], s=90, color=INK, zorder=4, label="portfolio")
TQ = np.geomspace(P[:, 1].min() * 0.9, P[:, 1].max() * 1.1, 50)
for sc in (3e8, 3.3e8, 3.7e8):
    ax.plot(TQ, sc / TQ, color=RULE, lw=1, zorder=1)
cb = plt.colorbar(sc_, ax=ax, pad=0.01); cb.set_label("release day"); cb.ax.set_ylim(0, dnum.max()); cb.set_ticks([0, dnum.max()]); cb.set_ticklabels([str(days.min()), str(days.max())])
ax.set_xscale("log"); ax.set_xlim(P[:, 1].min() * 0.97, np.percentile(P[:, 1], 95) * 1.05)
ax.xaxis.set_major_formatter(FuncFormatter(lambda x, _: f"{x / 1e6:.1f}M")); ax.xaxis.set_minor_formatter(FuncFormatter(lambda x, _: f"{x / 1e6:.1f}M")); ax.set_xlabel("T x Q of the block encoding"); ax.set_ylabel("lambda_eff (Ha)")
ax.set_ylim(P[:, 0].min() - 0.3, np.percentile(P[:, 0], 97) + 0.3); ax.legend(loc="upper right", fontsize=10)
headline(fig, "Chemistry landscape: lambda_eff fell first, then T x Q", f"{len(P)} public submissions (manwithacat's dataset); darker = later; grey = equal cost")
plt.show()

<a id="s7"></a>
<div style="font-family: 'Segoe UI', Roboto, 'Helvetica Neue', Arial, sans-serif; background: #0b1f4b; color: #ffffff; padding: 10px 16px; border-radius: 8px; font-size: 22px; font-weight: 700;">6 &middot; The public lineage: how the frontier was built, and by whom</div>

This competition is a relay race in public. manwithacat's *VQCC public circuits* dataset records, for every public
circuit, its release time, its publisher and its cost under the official verifier. The best public ratio per task by
release time shows where the community found its gains; the labels name the publishers of the largest steps.

In [ ]:
# Figure: the best public circuit per task over time, with the largest steps labelled by publisher.
task_key = {"invsqrt": "invsqrt", "dqi": "dqi", "chemistry": "dfthcsos"}
series, steps = {}, {}
for t_, key in task_key.items():
    pts = sorted((r["release_date"][:16], fnum(r["dqi_zero_failure_cost"]) if t_ == "dqi" and r.get("dqi_zero_failure_cost") else fnum(r["local_task_score"]), r["author"]) for r in man if r["task"] == t_)
    best, out = math.inf, []
    for d, c, a in pts:
        if c == c and c < best: best = c; out.append((np.datetime64(d.replace(" ", "T")), REF[key] / c, a))
    series[t_] = out
fig, axs = plt.subplots(3, 1, figsize=(12, 9.2), sharex=True)
for ax, (t_, out) in zip(axs, series.items()):
    key = task_key[t_]; d, r, a = zip(*out); col = TC[key]
    ax.step(d, r, where="post", color=col); ax.scatter(d, r, s=16, color=col, zorder=3, edgecolor=SURF, linewidth=0.8)
    ax.axhline(ratios[key], color=INK, lw=1, ls=(0, (4, 3))); ax.text(d[0], ratios[key], f" portfolio {ratios[key]:.2f}x", va="bottom", fontsize=10, color=INK)
    jumps = sorted(range(1, len(r)), key=lambda i: r[i] / r[i - 1], reverse=True)[:4]
    for n_, i in enumerate(sorted(jumps)):
        ax.annotate(f"{a[i].split(' (')[0][:22]}\n+{100 * (r[i] / r[i - 1] - 1):.1f}%", (d[i], r[i]), (0, -22 - 24 * (n_ % 2)), textcoords="offset points", ha="center", va="top", fontsize=9.5, color=INK2,
                    arrowprops=dict(arrowstyle="-", color=MUTED, lw=0.6))
    lo, hi = min(r) * 0.92, max(max(r), ratios[key]) * 1.04
    ax.set_ylim(lo - (hi - lo) * 0.6, hi); ax.set_ylabel("ref / best cost"); ax.set_title(f"{TN[key]}: {len(out)} frontier steps", loc="left", fontsize=11.5, color=INK)
fig.autofmt_xdate()
headline(fig, "The public frontier, step by step: many small gains, a few big jumps",
         "best public ratio per task by release time (manwithacat's dataset); labels = publisher of the four largest steps; dashed = this portfolio")
plt.show()
for t_, out in series.items(): print(f"{t_:9s} {len(out):3d} frontier records; last at {str(out[-1][0])[:16]}: {out[-1][1]:.3f}x by {out[-1][2]}")

### The last days, after the dataset scan
The dataset behind the timeline ends on 2026-11-10; our [QCC Circuit Atlas](https://www.kaggle.com/datasets/asteriospanoras/qcc-circuit-atlas)
(scan of 2026-11-12) and the public notebooks since then add the jumps that built today's submission: the **46-qubit
invsqrt line** (Syed Asad Ali's own circuit, Ali Haydar &Ouml;zda&#287;'s `4560ac0c`, then `dcde7078`, 78,765.51 against
93,555.26 for the best 62-qubit splice), the **889-qubit DQI** (Syed Asad Ali's from-source v12 decoder at Ert&#287;rul
&Ouml;zer's cutoff 18 with Areeb Ali's 241 retained products, `821bf8d1`, cost 3.602e11 against 3.716e11 for
maoxin123's 892-qubit `4c81b726`), and maoxin123's **4,038-Toffoli coded-word chemistry loaders** with Heliosli's
weights. Heliosli's public notebook put these together at 27.2971; our one-Toffoli fix gave 27.3024; maoxin123's v27 loaders reached 27.30661 and our chemistry weight refit 27.3067 (local, default salt).

### Who moved the frontier
Counting, per publisher, the public files that set a new best for a task. It is a crude measure (one big idea and
one small re-tune both count as one, and a republication is credited to its publisher), but it shows how many people
the frontier depends on. Section 8 credits the ideas, not just the files.

In [ ]:
cnt = collections.defaultdict(collections.Counter)
for t_, out in series.items():
    for d, r, a in out: cnt[re.sub(r"^team ", "", re.sub(r"\s*\(.*\)$", "", a).strip())][task_key[t_]] += 1
people = sorted(cnt, key=lambda p: -sum(cnt[p].values()))
topn = people[:20]
fig, ax = plt.subplots(figsize=(12, 5.6))
left = np.zeros(len(topn))
for key in ("invsqrt", "dqi", "dfthcsos"):
    vals = np.array([cnt[p][key] for p in topn]); ax.barh(range(len(topn)), vals, left=left, color=TC[key], height=0.66, label=TN[key], edgecolor=SURF, linewidth=1.5); left += vals
for i, tot in enumerate(left): ax.text(tot + 0.2, i, f"{int(tot)}", va="center", fontsize=10, color=INK2)
ax.set_yticks(range(len(topn))); ax.set_yticklabels(topn); ax.invert_yaxis(); ax.legend(loc="lower right", fontsize=10.5); ax.grid(axis="y", visible=False)
ax.set_xlabel("public files that set a new best for a task")
headline(fig, f"{len(people)} publishers set at least one public best; the top 20 are shown",
         "frontier records from manwithacat's public-circuits dataset, by publisher and task")
plt.show()

### The family tree of the submission
A simplified map of what each file is built on, from the authors' own credit notes (section 8 has every name and
link). Arrows read *is built on / contributes to*.

In [ ]:
# Figure: the submission's family tree (curated from the authors' own credits; simplified).
nodes = {  # key: (x, y, task, label)
    "i1": (1, 4.4, "invsqrt", "Syed Asad Ali\nown-circuit invsqrt\n(46 qubits)"),
    "i2": (2, 4.4, "invsqrt", "Ali Haydar Özdağ\n4560ac0c"),
    "i3": (3, 4.95, "invsqrt", "dcde7078, shipped by\nAlex Chilton and\nAtakan Aldemir"),
    "i4": (4.3, 4.4, "invsqrt", "SUBMISSION invsqrt\ndcde7078 + our\ndead-write fix"),
    "d0": (0, 2.2, "dqi", "decoder line: jday96314,\nSomeJoint, team i want to\nrun gemma 9b, Heliosli, ..."),
    "d1": (1, 2.2, "dqi", "Syed Asad Ali\nfrom-source decoder\n(v7 -> v12)"),
    "d2": (2, 2.75, "dqi", "Ertuğrul Özer\nextractor cutoff\n17 / 18"),
    "d3": (2, 1.65, "dqi", "Areeb Ali\nproduct retention\n(241 products)"),
    "d4": (4.3, 2.2, "dqi", "SUBMISSION DQI\n821bf8d1\n(889 qubits)"),
    "c0": (0, 0.0, "dfthcsos", "Lera (marculera)\nDF-THC line and\nthe authors she credits"),
    "c1": (1, 0.0, "dfthcsos", "maoxin123 (8,143,44)\ntable, coded words,\n4,038-T loaders"),
    "c2": (2, -0.55, "dfthcsos", "prvsiyan\nrefit code"),
    "c3": (3, -0.55, "dfthcsos", "maoxin123 v27\nloaders 71735b23 +\nour weight refit"),
    "c4": (4.3, 0.0, "dfthcsos", "SUBMISSION chemistry\nv27 loaders +\nrefit weights"),
}
edges = [("i1", "i2"), ("i2", "i3"), ("i3", "i4"), ("d0", "d1"), ("d1", "d2"), ("d1", "d3"), ("d2", "d4"), ("d3", "d4"),
         ("c0", "c1"), ("c1", "c4"), ("c2", "c3"), ("c1", "c3"), ("c3", "c4")]
nodes = {k_: (x * 1.45, y_, t_, l_) for k_, (x, y_, t_, l_) in nodes.items()}
fig, ax = plt.subplots(figsize=(12.5, 6.8)); ax.set_xlim(-0.8, 7.1); ax.set_ylim(-1.3, 5.75); ax.axis("off")
bw, bh = 1.25, 0.78
for a_, b_ in edges:
    (x1, y1, *_), (x2, y2, *_) = nodes[a_], nodes[b_]
    ax.annotate("", (x2 - bw / 2, y2), (x1 + bw / 2, y1), arrowprops=dict(arrowstyle="-|>", color=INK2, lw=1, shrinkA=0, shrinkB=2))
for k_, (x, y_, task, lab) in nodes.items():
    pf = lab.startswith("SUBMISSION"); col = TC[task]
    ax.add_patch(FancyBboxPatch((x - bw / 2, y_ - bh / 2), bw, bh, boxstyle="round,pad=0.02", facecolor=col if pf else SURF, edgecolor=col, lw=2 if pf else 1.3))
    ax.text(x, y_, lab, ha="center", va="center", fontsize=9, color="white" if pf else INK, fontweight="bold" if pf else "normal")
for task, y_ in (("invsqrt", 5.55), ("dqi", 3.4), ("dfthcsos", 0.75)): ax.text(-0.75, y_, TN[task], fontsize=13, fontweight="bold", color=TC[task])
headline(fig, "The submission's family tree: three relay races, dozens of authors",
         "simplified, from the authors' own credit notes; filled boxes = the files in the submission")
plt.show()

<a id="s8"></a>
<div style="font-family: 'Segoe UI', Roboto, 'Helvetica Neue', Arial, sans-serif; background: #0b1f4b; color: #ffffff; padding: 10px 16px; border-radius: 8px; font-size: 22px; font-weight: 700;">7 &middot; Open problems (ideas welcome in the comments)</div>

<div style="font-family: 'Segoe UI', Roboto, 'Helvetica Neue', Arial, sans-serif; font-size: 15.5px; line-height: 1.6; color: #111111; background: #ffffff;">
Each of these survived the checks of section 3 as a <i>possible</i> gain, with the reason it is still open.
<ol style="padding-left: 22px;">
<li><b>invsqrt at 47 qubits?</b> The +1 qubit break-even is a cut of about 36 executed Toffolis. Product retention gives only 2, but the ~69
conditional carry fix-ups of the backward half are a pool large enough, if a third carry slot can remove most of them (untested; we rate it unlikely).</li>
<li><b>A better invsqrt approximation.</b> The rotations cannot be farmed (section 3), but each doubling of the phase margin is worth 4.6
Toffoli-equivalents, and one rotation fewer about 7: a table or interpolation design with more margin would pay.</li>
<li><b>DQI at 888 qubits.</b> The 509 blocking peak episodes of <code style="background: #eef1f6; color: #111111; padding: 1px 5px; border-radius: 4px; font-size: 92%;">821bf8d1</code> fall into seven families, each with a one-wire reallocation
pattern already used elsewhere in the builder (estimate +0.010 to +0.021 points). It needs the decoder's source build to write exactly.</li>
<li><b>A cheaper GF(2<sup>24</sup>) &rarr; GF(2<sup>12</sup>) norm.</b> The norm is a quadratic map computed today with the rank-42 bilinear product; its
multiplicative complexity may be lower. 42 &rarr; 36 would save about 1.2k Toffolis (0.3%, estimate).</li>
<li><b>An exact DQI decoder.</b> The rare failures cost about 0.019 points in expectation (estimate). Handling the abnormal Euclid steps exactly
for less than that would make the score salt-proof.</li>
<li><b>A chemistry rebuild at C = 40.</b> Same-pipeline evidence (Lera's (9, 143, C) builds) suggests &minus;1.8% cost if a C = 40 factorization
reaches the public table's quality, about +0.16 points (estimate). It needs a new factorization, new coded words and new loaders.</li>
<li><b>Fully mixed loader products.</b> The counting floor sits about 25 products per loader below today's method, if any construction can
reach it; ours could not.</li>
</ol>
If you test one of these, please post what happened, negative results included. We will credit every contribution.
</div>

<a id="s9"></a>
<div style="font-family: 'Segoe UI', Roboto, 'Helvetica Neue', Arial, sans-serif; background: #0b1f4b; color: #ffffff; padding: 10px 16px; border-radius: 8px; font-size: 22px; font-weight: 700;">8 &middot; Full credits and licences</div>

**invsqrt** <code style="background: #eef1f6; color: #111111; padding: 1px 5px; border-radius: 4px; font-size: 92%;">dcde7078</code> + our dead-write fix
- asteriospanoras: the dead-write fix (op 3292), found with <code style="background: #eef1f6; color: #111111; padding: 1px 5px; border-radius: 4px; font-size: 92%;">qccscope</code>, proved on all 3,133,452 inputs, official 78,765.51 &rarr; 78,719.53.
- <code style="background: #eef1f6; color: #111111; padding: 1px 5px; border-radius: 4px; font-size: 92%;">dcde7078</code>, as shipped in [Alex Chilton's portfolio](https://www.kaggle.com/code/alexchilton/24-94-lb-verified-three-task-portfolio) and
  [Atakan Aldemir's notebooks](https://www.kaggle.com/code/atakanaldemir/qcc-gap-3-residue-repair-and-c43-native-compiler): the 46-qubit
  own-circuit line of [Syed Asad Ali](https://www.kaggle.com/code/syed3000/vqcc-own-circuit-portfolio-notebook) and
  [Ali Haydar &Ouml;zda&#287;'s](https://www.kaggle.com/code/ahoirg/qcc-submit-verify-and-package) <code style="background: #eef1f6; color: #111111; padding: 1px 5px; border-radius: 4px; font-size: 92%;">4560ac0c</code>; its lineage is credited there.
- The 62-qubit line used in our earlier versions and in the landscape: Syed Asad Ali, Hosen42, Kirill Tymchishin, Atakan Aldemir,
  Ert&#287;rul &Ouml;zer, busyaprime, yoyobar, mrmeows / team selenium, Leo Tang, Ehan Mallick, maoxin123, Gengsr, manwithacat (HW-25 join),
  WCao24 (44-CSWAP normaliser) and Thomas Gamet (early prefix), whose two circuits our 12 November splice joined.

**DQI** <code style="background: #eef1f6; color: #111111; padding: 1px 5px; border-radius: 4px; font-size: 92%;">821bf8d1</code>
- [Areeb Ali](https://www.kaggle.com/code/areebbb/qai-fewer-toffolis-same-qubits-lb-27-11042): 241 retained products, on Syed Asad Ali's
  v12 from-source decoder at extractor cutoff 18 ([Ert&#287;rul &Ouml;zer](https://www.kaggle.com/code/ozertuu/quantum-ai-27-083-vqcc-dqi-cutoff-17)).
- The earlier decoder lineage (team *i want to run gemma 9b*, Heliosli, SomeJoint, jday96314, Kirill Tymchishin, Atakan Aldemir, Ali Haydar
  &Ouml;zda&#287;, Isabe11aM00re, MAIDANG, daifanhao, Oleg Sher, and the Khattar-Gidney paper the decoder follows) is credited in those notebooks.
- DQI files discussed as history: maoxin123's 392k-Toffoli rewrite <code style="background: #eef1f6; color: #111111; padding: 1px 5px; border-radius: 4px; font-size: 92%;">4c81b726</code> ([notebook](https://www.kaggle.com/code/maoxin123/qcc-v22-a-392k-toffoli-dqi-decoder));
  Areeb Ali's <code style="background: #eef1f6; color: #111111; padding: 1px 5px; border-radius: 4px; font-size: 92%;">8eccde65</code>; Atakan Aldemir's <code style="background: #eef1f6; color: #111111; padding: 1px 5px; border-radius: 4px; font-size: 92%;">eae396ba</code>, <code style="background: #eef1f6; color: #111111; padding: 1px 5px; border-radius: 4px; font-size: 92%;">bb4287e7</code> and the readout windows of <code style="background: #eef1f6; color: #111111; padding: 1px 5px; border-radius: 4px; font-size: 92%;">e9f54be3</code>
  ([Terminal Readout and Phase Folds](https://www.kaggle.com/code/atakanaldemir/qcc-terminal-readout-and-phase-folds)); Heliosli's <code style="background: #eef1f6; color: #111111; padding: 1px 5px; border-radius: 4px; font-size: 92%;">9fcda0c1</code>.
  Ours: the readout-window transplant onto <code style="background: #eef1f6; color: #111111; padding: 1px 5px; border-radius: 4px; font-size: 92%;">bb4287e7</code> (&minus;48 Toffolis per shot) and the 2-product retention on <code style="background: #eef1f6; color: #111111; padding: 1px 5px; border-radius: 4px; font-size: 92%;">4c81b726</code>.

**Chemistry** maoxin123's v27 table + loaders <code style="background: #eef1f6; color: #111111; padding: 1px 5px; border-radius: 4px; font-size: 92%;">71735b23</code>, <b>weights refitted by us</b>
- [maoxin123](https://www.kaggle.com/code/maoxin123/qcc-v27-4033-toffoli-coded-word-chemistry): the (8, 143, 44) table, coded words and v27 loaders.
- Weights by [Heliosli](https://www.kaggle.com/code/heliosli/the-dataset-is-public-now-enjoy) and
  [Atakan Aldemir](https://www.kaggle.com/code/atakanaldemir/qcc-gap-3-residue-repair-and-c43-native-compiler) with
  [prvsiyan's refit code](https://www.kaggle.com/datasets/prvsiyan/vqcc-verified-best-of-circuits-1106a).
- Lera (marculera)'s DF-THC line ([dfthc-c44-build](https://www.kaggle.com/datasets/marculera/dfthc-c44-build)) and the authors her NOTICE
  credits (Ahmed Berat &Ouml;zer, kozykappa, anazemcev, mrmeows, michaelrustad, phucthaiv02, somejoint, gengsr, qwertykeypadapple,
  lightningv08, syed3000, Heliosli, busyaprime).

**Practice and analysis.** Alex Chilton's [Verified Three-Task Portfolio](https://www.kaggle.com/code/alexchilton/24-94-lb-verified-three-task-portfolio)
(pinning by SHA-256, re-verifying with the stock verifiers); WCao24's byte-reproducible builds; Oleg Sher's
[Quantum Circuits from Zero](https://www.kaggle.com/code/sheroleg/quantum-circuits-from-zero-1-sqrt-x-explained).

**This notebook.** The experiments of section 3, <code style="background: #eef1f6; color: #111111; padding: 1px 5px; border-radius: 4px; font-size: 92%;">qccscope</code>, <code style="background: #eef1f6; color: #111111; padding: 1px 5px; border-radius: 4px; font-size: 92%;">kmsim</code> and the figures: asteriospanoras. The landscapes, the timeline and the
publisher counts come from manwithacat's [VQCC public circuits](https://www.kaggle.com/datasets/manwithacat/vqcc-apache-public-circuits)
dataset (Apache-2.0); the public circuits are also indexed in our [QCC Circuit Atlas](https://www.kaggle.com/datasets/asteriospanoras/qcc-circuit-atlas).
The method of section 4 borrows from Yasin (top-down analysis, ISPASS 2014), Gregg (USE method), Williams, Waterman and Patterson (roofline,
CACM 2009), Amdahl (1967), Massalin (1987), Schkufza et al. (STOKE, 2013), Xu et al. (Quartz 2022, QUESO 2023), Willsey et al. (egg, 2021),
Kissinger and van de Wetering (2020), Gidney (temporary AND, 2018; pebble games), Cuccaro et al. (2004), Khattar and Gidney (2024), and the
bilinear-rank survey of Ballet et al. (2019). Task descriptions follow the competition pages and the hosts' paper
([arXiv:2610.23832](https://arxiv.org/abs/2610.23832)).

Tasks, kickmix and verifiers: Google Quantum AI / Qualtran. Each circuit keeps its authors' terms; nothing here is relicensed. The code of
this notebook (including <code style="background: #eef1f6; color: #111111; padding: 1px 5px; border-radius: 4px; font-size: 92%;">kmsim</code> and <code style="background: #eef1f6; color: #111111; padding: 1px 5px; border-radius: 4px; font-size: 92%;">qccscope</code>) is Apache-2.0.

<div style="font-family: 'Segoe UI', Roboto, 'Helvetica Neue', Arial, sans-serif; font-size: 15.5px; background: linear-gradient(120deg, #0b1f4b 0%, #1d5aa6 100%); color: #ffffff; padding: 14px 18px; border-radius: 10px; margin-top: 8px;">
<b>Thank you for reading.</b> If this notebook helped you, an upvote helps others find it; corrections, negative results and ideas in the
comments make the next version better.
</div>

<a id="app"></a>
<div style="font-family: 'Segoe UI', Roboto, 'Helvetica Neue', Arial, sans-serif; background: #0b1f4b; color: #ffffff; padding: 10px 16px; border-radius: 8px; font-size: 22px; font-weight: 700;">Appendix &middot; the splice search (history, 62-qubit circuits)</div>

HW-25 (manwithacat) joined two public circuits where they agree on every wire. We generalised it: a cut is *clean*
when the condition stack is empty and no classical bit written before it is read after it; at a clean cut the rest of
the circuit depends only on the wires, and the phase so far is a function of x alone. Two circuits whose clean cuts
hold the same wires and phase on every sampled input can be joined there. A shortest path over all such sync points
of all public 62-qubit invsqrt circuits gives the cheapest splice.

**History.** On the public set of early November, this found nothing beyond HW-25. Once WCao24's normaliser
(<code style="background: #eef1f6; color: #111111; padding: 1px 5px; border-radius: 4px; font-size: 92%;">94b29cc5</code>) and Thomas Gamet's early prefix (<code style="background: #eef1f6; color: #111111; padding: 1px 5px; border-radius: 4px; font-size: 92%;">5ec4f516</code>) were public, the same search found our earlier
<code style="background: #eef1f6; color: #111111; padding: 1px 5px; border-radius: 4px; font-size: 92%;">9c708f7b</code>: 1,213.87 Toffolis per shot against 1,216.83 for the best single circuit (official verifier, -0.20% cost).
The cell below runs the search on whatever public set is attached, on a small and a large input sample. **Always
re-measure on a large fresh sample**: conditional blocks fire on random measurement outcomes, so on a small sample
the search picks segments that were cheap by chance.

In [ ]:
# Slow: a few minutes on Kaggle. clean cuts of every public 62-qubit circuit, then a shortest path.
def bits_last_read(ops):
    last = {}
    for k, (n, qs, bs, c, a) in enumerate(ops):
        if c is not None: last[c] = k
        if n == "BIT_INVERT": last[bs[0]] = k
    return last
def clean_cuts(path, xs, seed=7):
    ops, regs = parse(path)
    nb = 1 + max(max((max(o[2]) for o in ops if o[2]), default=0), max((o[3] for o in ops if o[3] is not None), default=0))
    last = bits_last_read(ops); s = Sim(ops, 62, nb, len(xs), seed)
    for i, qi in enumerate(regs["r0"]): s.q[qi] = (xs >> i) & 1 == 1
    H = lambda: hashlib.blake2b(np.packbits(s.q).tobytes() + s.ph.tobytes(), digest_size=16).hexdigest()
    cuts, open_until, depth, rot = [(0, H(), 0.0, 0)], -1, 0, 0
    for k, (n, qs, bs, c, a) in enumerate(ops):
        depth += (n == "PUSH_CONDITION") - (n == "POP_CONDITION")
        s.run(k, k + 1)
        for b in bs: open_until = max(open_until, last.get(b, -1))
        if n == "Z_POW" and a % (1 << 61): rot += 1
        if depth == 0 and open_until <= k: cuts.append((k + 1, H(), float(s.toff.mean()), rot))
    return cuts
def splice(data, rot_eq):
    by_hash = collections.defaultdict(list)
    for name, cuts in data.items():
        for i, c in enumerate(cuts): by_hash[c[1]].append((name, i))
    start = next(iter(data.values()))[0][1]; dist, pq, best = {}, [], None
    for node in by_hash[start]: dist[node] = 0.0; heapq.heappush(pq, (0.0, *node))
    while pq:
        d, name, i = heapq.heappop(pq)
        if d > dist.get((name, i), 1e18): continue
        cuts = data[name]
        if i == len(cuts) - 1:
            best = min(best or (1e18,), (d, name)); continue
        w_ = cuts[i + 1][2] - cuts[i][2] + rot_eq * (cuts[i + 1][3] - cuts[i][3])
        for nxt, nd in [((name, i + 1), d + w_)] + [(o, d) for o in by_hash[cuts[i][1]] if o != (name, i)]:
            if nd < dist.get(nxt, 1e18): dist[nxt] = nd; heapq.heappush(pq, (nd, *nxt))
    return best
cands = {k: p for k, p in PUB_KMX.items() if 1 + max(max(o[1]) for o in parse(p)[0] if o[1]) == 62}
log(len(PUB_KMX), "public invsqrt circuits,", len(cands), "with 62 qubits")
rot_eq = (0.56 * math.log2(1 / v["invsqrt"]["rotation_eps"]) + 5.3) / 4
for label, n_rand in (("small sample (512 shots)", 384), ("large sample (4,098 shots)", 3072)):
    xs_s = np.concatenate([np.arange(1, 1 + (128 if n_rand == 384 else 1024)), np.random.default_rng(11).integers(1, 3133453, n_rand), [161, 3133452]]).astype(np.int64)
    data = {k: clean_cuts(p, xs_s) for k, p in cands.items()}
    singles = {k: c[-1][2] + rot_eq * c[-1][3] for k, c in data.items()}
    best = splice(data, rot_eq); b1 = min(singles, key=singles.get)
    print(f"{label}: best single {b1} {singles[b1]:.3f}, best splice {best[0]:.3f} ({best[0] - singles[b1]:+.3f})")
log("splice done")

In [ ]:
log("done")